### Multi-Floor Maze — Tổng hợp môi trường & Huấn luyện
Thành viên nhóm 7:
Phan Quốc Viễn - 23110362
Nghiêm Quang Huy - 23110222

**Yêu cầu:** Cài `gymnasium`, `numpy`, `torch`, `stable-baselines3`, `Pillow`.


In [ ]:
# --- Cell 1: Thư viện + CURRICULUM ---

import copy
import heapq
import random
from collections import defaultdict, deque
from enum import IntEnum

import gymnasium as gym
import numpy as np
from gymnasium import spaces

CURRICULUM = {
    1: {
        "name": "Crawl",
        "width": 7, "height": 7, "n_floors": 1, "max_steps": 50,
        "agent_hp": 10, "agent_ammo": 0, "agent_max_ammo": 0, "agent_stamina": 3,
        "n_traps": 0, "n_health": 2, "n_ammo": 0,
        "enemies": [], "locks": [],
        "start_floor": 0, "goal_floor": 0,
    },
    2: {
        "name": "Walk",
        "width": 9, "height": 9, "n_floors": 1, "max_steps": 60,
        "agent_hp": 10, "agent_ammo": 0, "agent_max_ammo": 0, "agent_stamina": 3,
        "n_traps": 0, "n_health": 2, "n_ammo": 0,
        "enemies": [], "locks": [],
        "start_floor": 0, "goal_floor": 0,
    },
    3: {
        "name": "Trapper",
        "width": 9, "height": 9, "n_floors": 1, "max_steps": 70,
        "agent_hp": 15, "agent_ammo": 0, "agent_max_ammo": 0, "agent_stamina": 3,
        "n_traps": 2, "n_health": 3, "n_ammo": 0,
        "enemies": [], "locks": [],
        "start_floor": 0, "goal_floor": 0,
    },
    4: {
        "name": "Locksmith",
        "width": 11, "height": 11, "n_floors": 1, "max_steps": 100,
        "agent_hp": 15, "agent_ammo": 0, "agent_max_ammo": 0, "agent_stamina": 3,
        "n_traps": 1, "n_health": 3, "n_ammo": 0,
        "enemies": [], "locks": [{"k": 0, "d": 0}],
        "start_floor": 0, "goal_floor": 0,
    },
    5: {
        "name": "Climber",
        "width": 11, "height": 11, "n_floors": 2, "max_steps": 150,
        "agent_hp": 20, "agent_ammo": 0, "agent_max_ammo": 0, "agent_stamina": 3,
        "n_traps": 2, "n_health": 4, "n_ammo": 0,
        "enemies": [], "locks": [],
        "start_floor": 1, "goal_floor": 0,
    },
    6: {
        "name": "Scout",
        "width": 11, "height": 11, "n_floors": 2, "max_steps": 180,
        "agent_hp": 20, "agent_ammo": 5, "agent_max_ammo": 5, "agent_stamina": 3,
        "n_traps": 2, "n_health": 5, "n_ammo": 1,
        "enemies": [{"type": "patrol", "floor": 0}],
        "locks": [],
        "start_floor": 1, "goal_floor": 0,
    },
    7: {
        "name": "Operative",
        "width": 11, "height": 11, "n_floors": 2, "max_steps": 250,
        "agent_hp": 10, "agent_ammo": 5, "agent_max_ammo": 5, "agent_stamina": 5,
        "n_traps": 3, "n_health": 3, "n_ammo": 2,
        "enemies": [
            {"type": "chaser", "floor": 0},
            {"type": "patrol", "floor": 1},
        ],
        "locks": [{"k": 0, "d": 0}],
        "start_floor": 1, "goal_floor": 0,
    },
    8: {
        "name": "Elite",
        "width": 13, "height": 13, "n_floors": 2, "max_steps": 350,
        "agent_hp": 10, "agent_ammo": 6, "agent_max_ammo": 6, "agent_stamina": 5,
        "n_traps": 3, "n_health": 3, "n_ammo": 3,
        "enemies": [
            {"type": "sniper", "floor": 0},
            {"type": "chaser", "floor": 0},
            {"type": "patrol", "floor": 1},
        ],
        "locks": [{"k": 0, "d": 0}, {"k": 1, "d": 1}],
        "start_floor": 1, "goal_floor": 0,
    },
    9: {
        "name": "Master",
        "width": 13, "height": 13, "n_floors": 3, "max_steps": 500,
        "agent_hp": 10, "agent_ammo": 8, "agent_max_ammo": 8, "agent_stamina": 6,
        "n_traps": 3, "n_health": 4, "n_ammo": 4,
        "enemies": [
            {"type": "sniper", "floor": 0},
            {"type": "chaser", "floor": 1},
            {"type": "patrol", "floor": 2},
            {"type": "chaser", "floor": 2},
        ],
        "locks": [{"k": 0, "d": 0}, {"k": 1, "d": 1}, {"k": 2, "d": 2}],
        "start_floor": 2, "goal_floor": 0,
    },
}


In [ ]:
# --- Cell 2: Tile & DIR_DELTA ---

class Tile(IntEnum):
    FLOOR = 0
    WALL = 1
    STAIR_UP = 3
    STAIR_DOWN = 4
    GOAL = 5
    TRAP = 6
    START = 7
    DOOR = 10
    KEY = 20
    HEALTH_PACK = 30
    AMMO_PACK = 31


DIR_DELTA = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}


In [ ]:
# --- Cell 3: BSP — ProceduralMapGenerator ---

class _BSPNode:
    __slots__ = ("x", "y", "w", "h", "left", "right", "room")

    def __init__(self, x, y, w, h):
        self.x, self.y, self.w, self.h = x, y, w, h
        self.left = self.right = self.room = None

    def leaves(self):
        if not self.left and not self.right:
            return [self]
        out = []
        if self.left:
            out.extend(self.left.leaves())
        if self.right:
            out.extend(self.right.leaves())
        return out


class ProceduralMapGenerator:
    def __init__(self, seed=None):
        self.rng = random.Random(seed)
        self.np_rng = np.random.RandomState(
            seed if seed is not None else None
        )

    def _bsp_params(self, w, h):
        s = min(w, h)
        if s <= 9:
            return 3, 2
        if s <= 14:
            return 4, 3
        return 5, 4

    def _bsp_split(self, node, ml, depth=0, md=4):
        if depth >= md:
            return
        can_h = node.w >= ml * 2
        can_v = node.h >= ml * 2
        if not can_h and not can_v:
            return
        if can_h and can_v:
            horiz = self.rng.random() < 0.5
        else:
            horiz = can_h
        if horiz:
            sp = self.rng.randint(ml, node.w - ml)
            node.left = _BSPNode(node.x, node.y, sp, node.h)
            node.right = _BSPNode(node.x + sp, node.y, node.w - sp, node.h)
        else:
            sp = self.rng.randint(ml, node.h - ml)
            node.left = _BSPNode(node.x, node.y, node.w, sp)
            node.right = _BSPNode(node.x, node.y + sp, node.w, node.h - sp)
        self._bsp_split(node.left, ml, depth + 1, md)
        self._bsp_split(node.right, ml, depth + 1, md)

    def _carve_rooms(self, node):
        for leaf in node.leaves():
            min_rw = min(3, leaf.w)
            min_rh = min(3, leaf.h)
            max_rw = max(min_rw, leaf.w - 2)
            max_rh = max(min_rh, leaf.h - 2)
            rw = self.rng.randint(min_rw, max_rw)
            rh = self.rng.randint(min_rh, max_rh)
            rx = self.rng.randint(leaf.x, max(leaf.x, leaf.x + leaf.w - rw))
            ry = self.rng.randint(leaf.y, max(leaf.y, leaf.y + leaf.h - rh))
            leaf.room = (rx, ry, rw, rh)

    def _get_rooms(self, node):
        return [leaf.room for leaf in node.leaves() if leaf.room]

    def _sibling_edges(self, node, rooms):
        edges = []
        if node.left and node.right:
            lr = self._get_rooms(node.left)
            rr = self._get_rooms(node.right)
            if lr and rr:
                bd, bp = float("inf"), None
                for a in lr:
                    ac = (a[0] + a[2] // 2, a[1] + a[3] // 2)
                    for b in rr:
                        bc = (b[0] + b[2] // 2, b[1] + b[3] // 2)
                        d = abs(ac[0] - bc[0]) + abs(ac[1] - bc[1])
                        if d < bd:
                            bd, bp = d, (rooms.index(a), rooms.index(b))
                if bp:
                    edges.append(bp)
            edges.extend(self._sibling_edges(node.left, rooms))
            edges.extend(self._sibling_edges(node.right, rooms))
        return edges

    def _build_graph(self, root, rooms, add_loops=True):
        n = len(rooms)
        if n <= 1:
            return defaultdict(set)
        adj = defaultdict(set)
        for a, b in self._sibling_edges(root, rooms):
            adj[a].add(b)
            adj[b].add(a)
        visited = {0}
        q = deque([0])
        while q:
            c = q.popleft()
            for nb in adj[c]:
                if nb not in visited:
                    visited.add(nb)
                    q.append(nb)
        for i in range(n):
            if i not in visited:
                bd, bj = float("inf"), 0
                ic = (rooms[i][0] + rooms[i][2] // 2, rooms[i][1] + rooms[i][3] // 2)
                for j in visited:
                    jc = (
                        rooms[j][0] + rooms[j][2] // 2,
                        rooms[j][1] + rooms[j][3] // 2,
                    )
                    d = abs(ic[0] - jc[0]) + abs(ic[1] - jc[1])
                    if d < bd:
                        bd, bj = d, j
                adj[i].add(bj)
                adj[bj].add(i)
                visited.add(i)
                q2 = deque([i])
                while q2:
                    c2 = q2.popleft()
                    for nb in adj[c2]:
                        if nb not in visited:
                            visited.add(nb)
                            q2.append(nb)
        if add_loops:
            extra = []
            for i in range(n):
                for j in range(i + 1, n):
                    if j not in adj[i]:
                        ci = (
                            rooms[i][0] + rooms[i][2] // 2,
                            rooms[i][1] + rooms[i][3] // 2,
                        )
                        cj = (
                            rooms[j][0] + rooms[j][2] // 2,
                            rooms[j][1] + rooms[j][3] // 2,
                        )
                        extra.append(
                            (abs(ci[0] - cj[0]) + abs(ci[1] - cj[1]), i, j)
                        )
            extra.sort()
            pool = extra[: max(1, len(extra) // 3)]
            n_add = min(max(1, n // 4), len(pool))
            if pool:
                for _, i, j in self.rng.sample(pool, n_add):
                    adj[i].add(j)
                    adj[j].add(i)
        return adj

    def _carve_corridors(self, grid, rooms, graph):
        done = set()
        for i in graph:
            for j in graph[i]:
                e = (min(i, j), max(i, j))
                if e in done:
                    continue
                done.add(e)
                self._carve_one(grid, rooms[i], rooms[j])

    def _carve_one(self, grid, r1, r2):
        c1x, c1y = r1[0] + r1[2] // 2, r1[1] + r1[3] // 2
        c2x, c2y = r2[0] + r2[2] // 2, r2[1] + r2[3] // 2
        h, w = grid.shape
        if self.rng.random() < 0.5:
            self._hc(grid, c1x, c2x, c1y, w, h)
            self._vc(grid, c1y, c2y, c2x, w, h)
        else:
            self._vc(grid, c1y, c2y, c1x, w, h)
            self._hc(grid, c1x, c2x, c2y, w, h)

    def _hc(self, g, x1, x2, y, w, h):
        for x in range(min(x1, x2), max(x1, x2) + 1):
            if 0 < y < h - 1 and 0 < x < w - 1:
                g[y][x] = Tile.FLOOR

    def _vc(self, g, y1, y2, x, w, h):
        for y in range(min(y1, y2), max(y1, y2) + 1):
            if 0 < y < h - 1 and 0 < x < w - 1:
                g[y][x] = Tile.FLOOR

    def _critical_path(self, graph, start, goal, n):
        if start == goal:
            return [start]
        vis = {start}
        q = deque([(start, [start])])
        while q:
            cur, path = q.popleft()
            if cur == goal:
                return path
            for nb in sorted(graph[cur]):
                if nb not in vis:
                    vis.add(nb)
                    q.append((nb, path + [nb]))
        return list(range(n))

    def _assign_zones(self, rooms, graph, cpath, n_enemies):
        n = len(rooms)
        zones = {}
        cp_set = set(cpath)
        zones[cpath[0]] = "spawn"
        zones[cpath[-1]] = "exit"
        for i in range(n):
            if i not in cp_set:
                zones[i] = "treasure"
        placed = 0
        if len(cpath) > 3:
            mid = len(cpath) // 2
            for idx in range(max(1, mid - 1), min(len(cpath) - 1, mid + 2)):
                ri = cpath[idx]
                if ri not in zones and placed < max(1, n_enemies):
                    zones[ri] = "combat"
                    placed += 1
        for ri in cpath:
            if ri not in zones:
                zones[ri] = "passage"
        return zones

    def _pick_in_room(self, grid, room, used):
        rx, ry, rw, rh = room
        h, w = grid.shape
        cands = [
            (x, y)
            for y in range(max(1, ry), min(h - 1, ry + rh))
            for x in range(max(1, rx), min(w - 1, rx + rw))
            if grid[y][x] == Tile.FLOOR and (x, y) not in used
        ]
        if cands:
            c = self.rng.choice(cands)
            used.add(c)
            return c
        return self._pick_floor(grid, used)

    def _pick_floor(self, grid, exc=None):
        h, w = grid.shape
        ex = exc or set()
        tiles = [
            (x, y)
            for y in range(h)
            for x in range(w)
            if grid[y][x] == Tile.FLOOR and (x, y) not in ex
        ]
        if tiles:
            c = self.rng.choice(tiles)
            if exc is not None:
                exc.add(c)
            return c
        return (1, 1)

    def _collect_candidates(self, grid, rooms, zones, zone_names, used):
        """Danh sách ô FLOOR chưa dùng trong các room thuộc zone_names (đặt từng ô có kiểm soát)."""
        out = []
        for ri, r in enumerate(rooms):
            if zones.get(ri) not in zone_names:
                continue
            rx, ry, rw, rh = r
            h, w = grid.shape
            for y in range(max(1, ry), min(h - 1, ry + rh)):
                for x in range(max(1, rx), min(w - 1, rx + rw)):
                    if grid[y][x] == Tile.FLOOR and (x, y) not in used:
                        out.append((x, y))
        return out

    def _in_any_room(self, x, y, rooms):
        for r in rooms:
            if r[0] <= x < r[0] + r[2] and r[1] <= y < r[1] + r[3]:
                return True
        return False

    def _bfs_tile_path(self, grid, sx, sy, gx, gy):
        h, w = grid.shape
        parent = {(sx, sy): None}
        q = deque([(sx, sy)])
        while q:
            cx, cy = q.popleft()
            if cx == gx and cy == gy:
                path = []
                cur = (cx, cy)
                while cur is not None:
                    path.append(cur)
                    cur = parent[cur]
                path.reverse()
                return path
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and (nx, ny) not in parent
                    and int(grid[ny][nx]) != int(Tile.WALL)
                ):
                    parent[(nx, ny)] = (cx, cy)
                    q.append((nx, ny))
        return None

    def _bfs_tile_connected(self, grid, sx, sy, gx, gy, extra_block=None):
        h, w = grid.shape
        block = {int(Tile.WALL)}
        if extra_block:
            block = block | extra_block
        vis = {(sx, sy)}
        q = deque([(sx, sy)])
        while q:
            cx, cy = q.popleft()
            if cx == gx and cy == gy:
                return True
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and (nx, ny) not in vis
                    and int(grid[ny][nx]) not in block
                ):
                    vis.add((nx, ny))
                    q.append((nx, ny))
        return False

    def _bfs_reachable_set(self, floors, start, block_types):
        sx, sy, sf = start
        nf = len(floors)
        h, w = floors[0].shape
        su, sd = {}, {}
        for f in range(nf):
            for y in range(h):
                for x in range(w):
                    t = int(floors[f][y][x])
                    if t == int(Tile.STAIR_UP) and f + 1 < nf:
                        su[(x, y, f)] = (x, y, f + 1)
                    elif t == int(Tile.STAIR_DOWN) and f > 0:
                        sd[(x, y, f)] = (x, y, f - 1)
        vis = {(sx, sy, sf)}
        q = deque([(sx, sy, sf)])
        while q:
            cx, cy, cf = q.popleft()
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and (nx, ny, cf) not in vis
                ):
                    t = int(floors[cf][ny][nx])
                    if t != int(Tile.WALL) and t not in block_types:
                        vis.add((nx, ny, cf))
                        q.append((nx, ny, cf))
            for stair_dict in (su, sd):
                if (cx, cy, cf) in stair_dict:
                    dest = stair_dict[(cx, cy, cf)]
                    if dest not in vis:
                        vis.add(dest)
                        q.append(dest)
        return vis

    # Giới hạn số ô thử khi tìm choke (sinh nhanh hơn, logic tương đương)
    _MAX_CHOKE_CANDIDATES = 14

    def _find_single_tile_choke(self, grid, rooms, entry_x, entry_y, exit_x, exit_y, used):
        """Find a single corridor tile that disconnects entry from exit when blocked."""
        h, w = grid.shape
        block = {int(Tile.WALL), int(Tile.DOOR)}
        parent = {(entry_x, entry_y): None}
        q = deque([(entry_x, entry_y)])
        while q:
            cx, cy = q.popleft()
            if cx == exit_x and cy == exit_y:
                break
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and (nx, ny) not in parent
                    and int(grid[ny][nx]) not in block
                ):
                    parent[(nx, ny)] = (cx, cy)
                    q.append((nx, ny))
        if (exit_x, exit_y) not in parent:
            return None
        path = []
        cur = (exit_x, exit_y)
        while cur is not None:
            path.append(cur)
            cur = parent[cur]
        path.reverse()
        mid = path[1:-1]
        for cx, cy in mid[: self._MAX_CHOKE_CANDIDATES]:
            if (cx, cy) in used or self._in_any_room(cx, cy, rooms):
                continue
            old = int(grid[cy][cx])
            grid[cy][cx] = int(Tile.WALL)
            connected = self._bfs_tile_connected(grid, entry_x, entry_y, exit_x, exit_y)
            grid[cy][cx] = old
            if not connected:
                return (cx, cy)
        return None

    def _perpendicular_seal(self, grid, cx, cy, rooms, used):
        h, w = grid.shape
        best = None
        for axis in ["v", "h"]:
            seal = [(cx, cy)]
            if axis == "v":
                for d in [-1, 1]:
                    yy = cy + d
                    while (
                        0 < yy < h - 1
                        and int(grid[yy][cx]) == int(Tile.FLOOR)
                        and not self._in_any_room(cx, yy, rooms)
                    ):
                        seal.append((cx, yy))
                        yy += d
            else:
                for d in [-1, 1]:
                    xx = cx + d
                    while (
                        0 < xx < w - 1
                        and int(grid[cy][xx]) == int(Tile.FLOOR)
                        and not self._in_any_room(xx, cy, rooms)
                    ):
                        seal.append((xx, cy))
                        xx += d
            seal = [(x, y) for x, y in seal if (x, y) not in used]
            if seal and len(seal) <= 2:
                if best is None or len(seal) < len(best):
                    best = seal
        return best

    def _find_stair_toward(self, grid, df, target_floor, h, w):
        look_for = (
            int(Tile.STAIR_UP) if target_floor > df else int(Tile.STAIR_DOWN)
        )
        for yy in range(h):
            for xx in range(w):
                if int(grid[yy][xx]) == look_for:
                    return (xx, yy)
        for yy in range(h):
            for xx in range(w):
                if int(grid[yy][xx]) in (
                    int(Tile.STAIR_DOWN),
                    int(Tile.STAIR_UP),
                ):
                    return (xx, yy)
        return None

    def _place_lock(self, all_f, all_r, all_g, kf, df, start, goal, used_per_floor):
        """Đặt 1 cửa trên floor df và 1 chìa trên floor kf. used_per_floor: list of set (x,y) per floor."""
        grid = all_f[df]
        rooms = all_r[df]
        used_df = used_per_floor[df]
        used_kf = used_per_floor[kf]
        h, w = grid.shape
        sx, sy, sfr = start
        gx, gy, gfr = goal

        entry_x, entry_y = sx, sy
        if df != sfr:
            sp = self._find_stair_toward(grid, df, sfr, h, w)
            if sp:
                entry_x, entry_y = sp
            else:
                return False

        exit_x, exit_y = gx, gy
        if df != gfr:
            sp = self._find_stair_toward(grid, df, gfr, h, w)
            if sp:
                exit_x, exit_y = sp
            else:
                return False

        if entry_x == exit_x and entry_y == exit_y:
            return False

        tile_path = self._bfs_tile_path(
            grid, entry_x, entry_y, exit_x, exit_y
        )
        if not tile_path or len(tile_path) < 3:
            return False

        door_type = int(Tile.DOOR)
        key_type = int(Tile.KEY)

        choke = self._find_single_tile_choke(
            grid, rooms, entry_x, entry_y, exit_x, exit_y, used_df
        )
        if choke:
            cx, cy = choke
            grid[cy][cx] = door_type
            reachable = self._bfs_reachable_set(all_f, start, {door_type})
            key_placed = False
            krooms = list(all_r[kf])
            self.rng.shuffle(krooms)
            for kr in krooms:
                kcx, kcy = kr[0] + kr[2] // 2, kr[1] + kr[3] // 2
                if (kcx, kcy, kf) in reachable:
                    kt = self._pick_in_room(all_f[kf], kr, used_kf)
                    if (kt[0], kt[1], kf) in reachable:
                        all_f[kf][kt[1]][kt[0]] = key_type
                        used_kf.add(kt)
                        key_placed = True
                        break
            if not key_placed:
                kh, kw = all_f[kf].shape
                for y2 in range(1, kh - 1):
                    for x2 in range(1, kw - 1):
                        if (
                            (x2, y2, kf) in reachable
                            and int(all_f[kf][y2][x2]) == int(Tile.FLOOR)
                            and (x2, y2) not in used_kf
                        ):
                            all_f[kf][y2][x2] = key_type
                            used_kf.add((x2, y2))
                            key_placed = True
                            break
                    if key_placed:
                        break
            if key_placed:
                used_df.add((cx, cy))
                return True
            grid[cy][cx] = Tile.FLOOR

        for cx, cy in (tile_path[1:-1])[: self._MAX_CHOKE_CANDIDATES]:
            seal = self._perpendicular_seal(grid, cx, cy, rooms, used_df)
            if seal and len(seal) == 1:
                old = {(x, y): int(grid[y][x]) for x, y in seal}
                for x, y in seal:
                    grid[y][x] = door_type

                if self._bfs_tile_connected(
                    grid, entry_x, entry_y, exit_x, exit_y, {door_type}
                ):
                    for (x, y), t in old.items():
                        grid[y][x] = t
                    continue

                reachable = self._bfs_reachable_set(
                    all_f, start, {door_type}
                )
                key_placed = False
                krooms = list(all_r[kf])
                self.rng.shuffle(krooms)
                for kr in krooms:
                    kcx, kcy = kr[0] + kr[2] // 2, kr[1] + kr[3] // 2
                    if (kcx, kcy, kf) in reachable:
                        kt = self._pick_in_room(all_f[kf], kr, used_kf)
                        if (kt[0], kt[1], kf) in reachable:
                            all_f[kf][kt[1]][kt[0]] = key_type
                            used_kf.add(kt)
                            key_placed = True
                            break

                if not key_placed:
                    kh, kw = all_f[kf].shape
                    for y2 in range(1, kh - 1):
                        for x2 in range(1, kw - 1):
                            if (
                                (x2, y2, kf) in reachable
                                and int(all_f[kf][y2][x2]) == int(Tile.FLOOR)
                                and (x2, y2) not in used_kf
                            ):
                                all_f[kf][y2][x2] = key_type
                                used_kf.add((x2, y2))
                                key_placed = True
                                break
                        if key_placed:
                            break

                if not key_placed:
                    for (x, y), t in old.items():
                        grid[y][x] = t
                    continue

                for x, y in seal:
                    used_df.add((x, y))
                return True

        return False

    def _bfs_multi(self, floors, start, goal):
        sx, sy, sf = start
        gx, gy, gf = goal
        nf = len(floors)
        h, w = floors[0].shape
        su, sd = {}, {}
        for f in range(nf):
            for y in range(h):
                for x in range(w):
                    if floors[f][y][x] == Tile.STAIR_UP and f + 1 < nf:
                        su[(x, y, f)] = (x, y, f + 1)
                    elif floors[f][y][x] == Tile.STAIR_DOWN and f > 0:
                        sd[(x, y, f)] = (x, y, f - 1)
        vis = set()
        q = deque([(sx, sy, sf, 0)])
        vis.add((sx, sy, sf))
        while q:
            cx, cy, cf, cd = q.popleft()
            if cx == gx and cy == gy and cf == gf:
                return cd
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and (nx, ny, cf) not in vis
                    and int(floors[cf][ny][nx]) != Tile.WALL
                ):
                    vis.add((nx, ny, cf))
                    q.append((nx, ny, cf, cd + 1))
            for sd_ in (su, sd):
                if (cx, cy, cf) in sd_:
                    dest = sd_[(cx, cy, cf)]
                    if dest not in vis:
                        vis.add(dest)
                        q.append((*dest, cd + 1))
        return -1

    def validate(self, map_data, config):
        floors = map_data["floors"]
        nf = map_data["n_floors"]
        w, h = map_data["width"], map_data["height"]
        sx, sy, sf = map_data["start"]
        gx, gy, gf = map_data["goal"]
        for f in range(nf):
            if np.sum(floors[f] != Tile.WALL) / (w * h) < 0.20:
                return False, "low_floor_ratio"
        dist = self._bfs_multi(floors, (sx, sy, sf), (gx, gy, gf))
        if dist < 0:
            return False, "unreachable"
        if dist < max(w, h) // 2:
            return False, "too_short"
        if config.get("locks"):
            # 1) Với tất cả cửa bị chặn, mọi chìa khoá phải vẫn đi tới được
            reachable = self._bfs_reachable_set(
                floors, (sx, sy, sf), {int(Tile.DOOR)}
            )
            for f in range(nf):
                for y in range(h):
                    for x in range(w):
                        if int(floors[f][y][x]) == int(Tile.KEY) and (
                            x,
                            y,
                            f,
                        ) not in reachable:
                            return False, "key_behind_door"
            # 2) Các floor có cửa: muốn tới stair/goal trên floor đó thì BẮT BUỘC phải qua ít
            #    nhất một cửa. Nghĩa là nếu xem mọi DOOR như tường, start không được chạm tới
            #    bất kỳ ô stair/goal nào trên những floor có lock.
            critical_targets = set()
            door_floors = {lock.get("d", 0) for lock in config.get("locks", [])}
            for f in door_floors:
                grid = floors[f]
                for y in range(h):
                    for x in range(w):
                        t = int(grid[y][x])
                        if t in (int(Tile.STAIR_UP), int(Tile.STAIR_DOWN)):
                            critical_targets.add((x, y, f))
                        if f == gf and t == int(Tile.GOAL):
                            critical_targets.add((x, y, f))
            if any(t in reachable for t in critical_targets):
                return False, "door_not_blocking_critical"
        return True, "ok"

    def _gen_floor(self, w, h, add_loops=True):
        grid = np.full((h, w), Tile.WALL, dtype=np.int32)
        ml, md = self._bsp_params(w, h)
        root = _BSPNode(1, 1, w - 2, h - 2)
        self._bsp_split(root, ml, 0, md)
        self._carve_rooms(root)
        rooms = self._get_rooms(root)
        if len(rooms) < 2:
            rw1 = max(3, (w - 2) // 3)
            rh1 = max(3, (h - 2) // 3)
            rw2 = max(3, (w - 2) // 3)
            rh2 = max(3, (h - 2) // 3)
            rooms = [
                (1, 1, rw1, rh1),
                (w - rw2 - 1, h - rh2 - 1, rw2, rh2),
            ]
        for rx, ry, rw, rh in rooms:
            for y in range(ry, min(ry + rh, h - 1)):
                for x in range(rx, min(rx + rw, w - 1)):
                    if 0 < y < h - 1 and 0 < x < w - 1:
                        grid[y][x] = Tile.FLOOR
        graph = self._build_graph(root, rooms, add_loops=add_loops)
        self._carve_corridors(grid, rooms, graph)
        return grid, rooms, graph

    def generate(self, config, max_retries=30):
        md = None
        for _ in range(max_retries):
            md = self._generate_once(config)
            ok, _ = self.validate(md, config)
            if ok:
                return md
        return md

    def _generate_once(self, config):
        nf = config.get("n_floors", 1)
        w = config.get("width", 12)
        h = config.get("height", 12)
        has_doors = bool(config.get("locks"))

        # BSP: giữ nguyên, chỉ sinh cấu trúc phòng/hành lang
        all_f, all_r, all_g = [], [], []
        for _ in range(nf):
            grid, rooms, graph = self._gen_floor(
                w, h, add_loops=not has_doors
            )
            all_f.append(grid)
            all_r.append(rooms)
            all_g.append(graph)

        # Mỗi floor một set ô đã dùng, tránh trùng và dễ kiểm soát
        used_per_floor = [set() for _ in range(nf)]

        # ─── 1) Cầu thang: từng cặp tầng, một vị trí nối fi ↔ fi+1 ───
        for fi in range(nf - 1):
            bd, stx, sty = float("inf"), w // 2, h // 2
            for ra in all_r[fi]:
                ac = (ra[0] + ra[2] // 2, ra[1] + ra[3] // 2)
                for rb in all_r[fi + 1]:
                    bc = (rb[0] + rb[2] // 2, rb[1] + rb[3] // 2)
                    d = abs(ac[0] - bc[0]) + abs(ac[1] - bc[1])
                    if d < bd:
                        bd = d
                        stx = max(1, min(w - 2, ac[0]))
                        sty = max(1, min(h - 2, ac[1]))
            all_f[fi][sty][stx] = Tile.STAIR_UP
            all_f[fi + 1][sty][stx] = Tile.STAIR_DOWN
            used_per_floor[fi].add((stx, sty))
            used_per_floor[fi + 1].add((stx, sty))

        sf = config.get("start_floor", nf - 1)
        gf = config.get("goal_floor", 0)

        # ─── 2) Start: một ô trên start_floor ───
        sr = all_r[sf]
        sp = (
            self._pick_in_room(all_f[sf], sr[0], used_per_floor[sf])
            if sr
            else self._pick_floor(all_f[sf], used_per_floor[sf])
        )
        used_per_floor[sf].add(sp)

        # ─── 3) Goal: một ô trên goal_floor, xa start ───
        gr = all_r[gf]
        if gr:
            bd2, br = -1, gr[0]
            for r in gr:
                rc = (r[0] + r[2] // 2, r[1] + r[3] // 2)
                d = abs(rc[0] - sp[0]) + abs(rc[1] - sp[1])
                if sf != gf:
                    d += w
                if d > bd2:
                    bd2, br = d, r
            gp = self._pick_in_room(all_f[gf], br, used_per_floor[gf])
        else:
            gp = self._pick_floor(all_f[gf], used_per_floor[gf])
        used_per_floor[gf].add(gp)

        all_f[sf][sp[1]][sp[0]] = Tile.START
        all_f[gf][gp[1]][gp[0]] = Tile.GOAL
        start = (sp[0], sp[1], sf)
        goal = (gp[0], gp[1], gf)

        # ─── 4) Cửa + chìa: từng lock một ───
        for lock in config.get("locks", []):
            kf = lock.get("k", 0)
            df = lock.get("d", 0)
            ok = self._place_lock(
                all_f, all_r, all_g, kf, df, start, goal, used_per_floor
            )
            if not ok:
                used_df = used_per_floor[df]
                used_kf = used_per_floor[kf]
                kr = all_r[kf]
                dr = all_r[df]
                if kr:
                    kt = self._pick_in_room(
                        all_f[kf],
                        kr[self.rng.randint(0, len(kr) - 1)],
                        used_kf,
                    )
                else:
                    kt = self._pick_floor(all_f[kf], used_kf)
                all_f[kf][kt[1]][kt[0]] = Tile.KEY
                if dr:
                    dt = self._pick_in_room(
                        all_f[df],
                        dr[self.rng.randint(0, len(dr) - 1)],
                        used_df,
                    )
                else:
                    dt = self._pick_floor(all_f[df], used_df)
                all_f[df][dt[1]][dt[0]] = Tile.DOOR

        # ─── 5) Bẫy / máu / đạn: đặt từng ô một, có zone và không trùng ───
        for fi in range(nf):
            grid = all_f[fi]
            rooms = all_r[fi]
            graph = all_g[fi]
            used = used_per_floor[fi]

            if len(rooms) < 2:
                for _ in range(config.get("n_traps", 0)):
                    t = self._pick_floor(grid, used)
                    if grid[t[1]][t[0]] == Tile.FLOOR:
                        grid[t[1]][t[0]] = Tile.TRAP
                for _ in range(config.get("n_health", 0)):
                    t = self._pick_floor(grid, used)
                    if grid[t[1]][t[0]] == Tile.FLOOR:
                        grid[t[1]][t[0]] = Tile.HEALTH_PACK
                for _ in range(config.get("n_ammo", 0)):
                    t = self._pick_floor(grid, used)
                    if grid[t[1]][t[0]] == Tile.FLOOR:
                        grid[t[1]][t[0]] = Tile.AMMO_PACK
                continue

            si, gi = 0, len(rooms) - 1
            if fi == sf:
                for ri, r in enumerate(rooms):
                    if r[0] <= sp[0] < r[0] + r[2] and r[1] <= sp[1] < r[1] + r[3]:
                        si = ri
                        break
            if fi == gf:
                for ri, r in enumerate(rooms):
                    if r[0] <= gp[0] < r[0] + r[2] and r[1] <= gp[1] < r[1] + r[3]:
                        gi = ri
                        break
            ne = sum(1 for ec in config.get("enemies", []) if ec.get("floor", 0) == fi)
            cp = self._critical_path(graph, si, gi, len(rooms))
            zones = self._assign_zones(rooms, graph, cp, ne)

            n_traps = config.get("n_traps", 0)
            n_health = config.get("n_health", 0)
            n_ammo = config.get("n_ammo", 0)
            # Mỗi loại build danh sách ô hợp lệ 1 lần rồi shuffle + pop → nhanh hơn
            trap_cands = self._collect_candidates(
                grid, rooms, zones, ("passage", "combat", "treasure"), used
            )
            self.rng.shuffle(trap_cands)
            for _ in range(n_traps):
                if trap_cands:
                    c = trap_cands.pop()
                    grid[c[1]][c[0]] = Tile.TRAP
                    used.add(c)
            health_cands = self._collect_candidates(
                grid, rooms, zones, ("treasure", "passage", "spawn"), used
            )
            self.rng.shuffle(health_cands)
            for _ in range(n_health):
                if health_cands:
                    c = health_cands.pop()
                    grid[c[1]][c[0]] = Tile.HEALTH_PACK
                    used.add(c)
            ammo_cands = self._collect_candidates(
                grid, rooms, zones, ("treasure", "combat", "passage"), used
            )
            self.rng.shuffle(ammo_cands)
            for _ in range(n_ammo):
                if ammo_cands:
                    c = ammo_cands.pop()
                    grid[c[1]][c[0]] = Tile.AMMO_PACK
                    used.add(c)

        return {
            "floors": all_f,
            "n_floors": nf,
            "width": w,
            "height": h,
            "start": start,
            "goal": goal,
        }



In [ ]:
# --- Cell 4: Entity system (Projectile, enemies) ---

class Projectile:
    __slots__ = ("x", "y", "floor", "dx", "dy", "owner", "ric", "alive")

    def __init__(self, x, y, floor, dx, dy, owner="agent", ric=0):
        self.x, self.y, self.floor = x, y, floor
        self.dx, self.dy = dx, dy
        self.owner = owner
        self.ric = ric
        self.alive = True

    def move(self):
        self.x += self.dx
        self.y += self.dy


class PatrolEnemy:
    def __init__(self, x, y, floor, waypoints=None, vision=4):
        self.x, self.y, self.floor = x, y, floor
        self.etype = "patrol"
        self.hp = 1
        self.alive = True
        self.wps = waypoints or [(x, y)]
        self.wp_idx = 0
        self.vision = vision
        self.state = "patrol"
        self.inv_target = None
        self.inv_timer = 0
        self.old_x, self.old_y = x, y

    def update(self, grid, apos, afloor, noise, npos):
        if not self.alive:
            return
        self.old_x, self.old_y = self.x, self.y
        if self.floor != afloor:
            self._patrol(grid)
            return
        ax, ay = apos
        blocked = {(ax, ay)}
        see = self._los(grid, ax, ay) and self._d(ax, ay) <= self.vision
        hear = (
            noise > 3
            and npos
            and abs(npos[0] - self.x) + abs(npos[1] - self.y) <= noise
        )
        if see:
            self.state = "investigate"
            self.inv_target = (ax, ay)
            self.inv_timer = 10
        elif hear:
            self.state = "investigate"
            self.inv_target = npos
            self.inv_timer = 8
        if self.state == "patrol":
            self._patrol(grid, blocked)
        elif self.state == "investigate":
            if self.inv_target:
                self._toward(grid, *self.inv_target, blocked=blocked)
            self.inv_timer -= 1
            if self.inv_timer <= 0:
                self.state = "patrol"

    def _patrol(self, grid, blocked=None):
        if not self.wps:
            return
        tx, ty = self.wps[self.wp_idx]
        if self.x == tx and self.y == ty:
            self.wp_idx = (self.wp_idx + 1) % len(self.wps)
            tx, ty = self.wps[self.wp_idx]
        self._toward(grid, tx, ty, blocked=blocked)

    def _toward(self, grid, tx, ty, blocked=None):
        bd = abs(tx - self.x) + abs(ty - self.y)
        best = None
        h, w = grid.shape
        for dx, dy in [(0, -1), (0, 1), (-1, 0), (1, 0)]:
            nx, ny = self.x + dx, self.y + dy
            if (
                0 <= nx < w
                and 0 <= ny < h
                and grid[ny][nx] not in (Tile.WALL, Tile.DOOR)
            ):
                if blocked and (nx, ny) in blocked:
                    continue
                d = abs(tx - nx) + abs(ty - ny)
                if d < bd:
                    bd, best = d, (nx, ny)
        if best:
            self.x, self.y = best

    def _los(self, grid, tx, ty):
        x0, y0 = self.x, self.y
        ddx, ddy = abs(tx - x0), abs(ty - y0)
        sx = 1 if tx > x0 else -1
        sy = 1 if ty > y0 else -1
        err = ddx - ddy
        h, w = grid.shape
        for _ in range(max(ddx, ddy) + 2):
            if x0 == tx and y0 == ty:
                return True
            if not (0 <= x0 < w and 0 <= y0 < h):
                return False
            if grid[y0][x0] in (Tile.WALL, Tile.DOOR) and (
                x0,
                y0,
            ) != (self.x, self.y):
                return False
            e2 = 2 * err
            if e2 > -ddy:
                err -= ddy
                x0 += sx
            if e2 < ddx:
                err += ddx
                y0 += sy
        return False

    def _d(self, ax, ay):
        return abs(ax - self.x) + abs(ay - self.y)


class ChaserEnemy:
    def __init__(self, x, y, floor, det=6):
        self.x, self.y, self.floor = x, y, floor
        self.etype = "chaser"
        self.hp = 2
        self.alive = True
        self.det = det
        self.state = "idle"
        self.timer = 0
        self.old_x, self.old_y = x, y

    def update(self, grid, apos, afloor, noise, npos):
        if not self.alive or self.floor != afloor:
            return
        self.old_x, self.old_y = self.x, self.y
        ax, ay = apos
        dist = abs(ax - self.x) + abs(ay - self.y)
        if dist <= self.det:
            self.state = "chase"
            self.timer = 15
        elif (
            noise > 5
            and npos
            and abs(npos[0] - self.x) + abs(npos[1] - self.y) <= noise
        ):
            self.state = "chase"
            self.timer = 10
        if self.state == "chase":
            path = self._astar(grid, (self.x, self.y), (ax, ay))
            if path and len(path) > 1:
                nx, ny = path[1]
                if (nx, ny) != (ax, ay):
                    self.x, self.y = nx, ny
            self.timer -= 1
            if self.timer <= 0:
                self.state = "idle"

    def _astar(self, grid, start, goal):
        h, w = grid.shape
        op = [(0, start)]
        came = {}
        gs = {start: 0}
        while op:
            _, cur = heapq.heappop(op)
            if cur == goal:
                p = [cur]
                while cur in came:
                    cur = came[cur]
                    p.append(cur)
                return p[::-1]
            for ddx, ddy in [(0, -1), (0, 1), (-1, 0), (1, 0)]:
                nx, ny = cur[0] + ddx, cur[1] + ddy
                if (
                    0 <= nx < w
                    and 0 <= ny < h
                    and grid[ny][nx] not in (Tile.WALL, Tile.DOOR)
                ):
                    tg = gs[cur] + 1
                    if (nx, ny) not in gs or tg < gs[(nx, ny)]:
                        gs[(nx, ny)] = tg
                        heapq.heappush(
                            op,
                            (
                                tg
                                + abs(nx - goal[0])
                                + abs(ny - goal[1]),
                                (nx, ny),
                            ),
                        )
                        came[(nx, ny)] = cur
        return None


class SniperEnemy:
    def __init__(self, x, y, floor):
        self.x, self.y, self.floor = x, y, floor
        self.etype = "sniper"
        self.hp = 1
        self.alive = True
        self.dir = (1, 0)
        self.charge = 0
        self.cd = 0
        self.laser_path = []
        self.charging = False
        self.old_x, self.old_y = x, y

    def update(self, grid, apos, afloor, noise, npos):
        self.laser_path = []
        proj = None
        if not self.alive or self.floor != afloor:
            return proj
        self.old_x, self.old_y = self.x, self.y
        if self.cd > 0:
            self.cd -= 1
            self.charging = False
            return proj
        ax, ay = apos
        dx, dy = ax - self.x, ay - self.y
        if abs(dx) >= abs(dy) and dx != 0:
            self.dir = (1 if dx > 0 else -1, 0)
        elif dy != 0:
            self.dir = (0, 1 if dy > 0 else -1)
        in_line = (self.dir[0] != 0 and dy == 0 and dx * self.dir[0] > 0) or (
            self.dir[1] != 0 and dx == 0 and dy * self.dir[1] > 0
        )
        if in_line:
            self.charging = True
            self.charge += 1
            h, w = grid.shape
            lx, ly = self.x, self.y
            while True:
                lx += self.dir[0]
                ly += self.dir[1]
                if (
                    not (0 <= lx < w and 0 <= ly < h)
                    or grid[ly][lx] in (Tile.WALL, Tile.DOOR)
                ):
                    break
                self.laser_path.append((lx, ly))
            if self.charge >= 3:
                proj = Projectile(
                    self.x,
                    self.y,
                    self.floor,
                    self.dir[0],
                    self.dir[1],
                    "sniper",
                    0,
                )
                self.charge = 0
                self.cd = 3
                self.charging = False
        else:
            self.charge = max(0, self.charge - 1)
            self.charging = False
        return proj



In [ ]:
# --- Cell 5: AgentState ---
class AgentState:
    def __init__(self, x, y, floor, hp=3, ammo=3, stamina=3):
        self.x, self.y, self.floor = x, y, floor
        self.hp, self.max_hp = hp, hp
        self.ammo, self.max_ammo = ammo, max(5, ammo)
        self.stamina, self.max_stamina = stamina, stamina
        self.noise_level = 0
        self.keys = 0
        self.last_dir = (1, 0)
        self.alive = True
        self.visited = set()
        self.steps = 0
        self.old_x, self.old_y = x, y

    def damage(self, n=1):
        self.hp -= n
        self.alive = self.hp > 0

    def heal(self, n=1):
        self.hp = min(self.max_hp, self.hp + n)

    def add_ammo(self, n=2):
        self.ammo = min(self.max_ammo, self.ammo + n)

    def use_ammo(self):
        if self.ammo > 0:
            self.ammo -= 1
            return True
        return False

    def use_stamina(self):
        if self.stamina > 0:
            self.stamina -= 1
            return True
        return False

    def regen_stamina(self):
        self.stamina = min(self.max_stamina, self.stamina + 1)


In [ ]:
# --- Cell 6: Physics & Mechanicsm ---

class BallisticsEngine:
    def __init__(self):
        self.bullets = []

    def fire(self, x, y, f, dx, dy, owner="agent", ric=1):
        self.bullets.append(Projectile(x, y, f, dx, dy, owner, ric))

    def add(self, p):
        if p:
            self.bullets.append(p)

    def update(self, grid, enemies, agent):
        events = []
        alive = []
        h, w = grid.shape
        for b in self.bullets:
            if not b.alive:
                continue
            b.move()
            if not (0 <= b.x < w and 0 <= b.y < h):
                b.alive = False
                continue
            if grid[b.y][b.x] in (Tile.WALL, Tile.DOOR):
                if b.ric > 0:
                    b.ric -= 1
                    if (
                        0 <= b.x - b.dx < w
                        and grid[b.y][b.x - b.dx]
                        not in (Tile.WALL, Tile.DOOR)
                    ):
                        b.dx = -b.dx
                    else:
                        b.dy = -b.dy
                    b.x += b.dx
                    b.y += b.dy
                    if not (0 <= b.x < w and 0 <= b.y < h) or grid[b.y][
                        b.x
                    ] in (Tile.WALL, Tile.DOOR):
                        b.alive = False
                else:
                    b.alive = False
                if b.alive:
                    alive.append(b)
                continue
            if b.owner == "agent":
                for e in enemies:
                    if (
                        e.alive
                        and e.x == b.x
                        and e.y == b.y
                        and e.floor == b.floor
                    ):
                        e.hp -= 1
                        if e.hp <= 0:
                            e.alive = False
                            events.append(("kill", e))
                        b.alive = False
                        break
            elif (
                b.x == agent.x
                and b.y == agent.y
                and b.floor == agent.floor
            ):
                events.append(("agent_hit", b))
                b.alive = False
            if b.alive:
                alive.append(b)
        self.bullets = alive
        return events

    def positions(self, f):
        return [
            (b.x, b.y, b.owner)
            for b in self.bullets
            if b.alive and b.floor == f
        ]


class StealthSystem:
    def __init__(self):
        self.noise = 0
        self.npos = None

    def calc(self, action, agent):
        if action == 8:
            n = 5
        elif 4 <= action <= 7:
            n = 10
        else:
            n = 0
        self.noise = n
        self.npos = (agent.x, agent.y) if n > 0 else None
        agent.noise_level = n
        return n

    def decay(self, agent):
        agent.noise_level = max(0, agent.noise_level - 2)


In [ ]:
# --- Cell 7: Gym obs (N_CHANNELS, _build_obs) ---

N_CHANNELS = 16
N_VEC = 15


def _build_obs(grid, agent, enemies, bpos, lasers, nf, target_3d, max_steps, vr=4):
    """Build (16, V, V) visual + (15,) vector observation."""
    vs = 2 * vr + 1
    h, w = grid.shape
    obs = np.zeros((N_CHANNELS, vs, vs), dtype=np.float32)

    for dy in range(-vr, vr + 1):
        for dx in range(-vr, vr + 1):
            gx, gy = agent.x + dx, agent.y + dy
            oy, ox = dy + vr, dx + vr
            if 0 <= gx < w and 0 <= gy < h:
                t = int(grid[gy][gx])
                ch_map = {
                    int(Tile.WALL): 0,
                    int(Tile.TRAP): 1,
                    int(Tile.DOOR): 2,
                    int(Tile.STAIR_UP): 3,
                    int(Tile.STAIR_DOWN): 4,
                    int(Tile.GOAL): 5,
                    int(Tile.KEY): 6,
                    int(Tile.HEALTH_PACK): 7,
                    int(Tile.AMMO_PACK): 8,
                }
                if t in ch_map:
                    obs[ch_map[t]][oy][ox] = 1.0
                if (gx, gy, agent.floor) in agent.visited:
                    obs[15][oy][ox] = 1.0
            else:
                obs[0][oy][ox] = 1.0

    obs[9][vr][vr] = 1.0

    ETYPE_CH = {"patrol": 10, "chaser": 11, "sniper": 12}
    min_enemy_dist = float("inf")
    for e in enemies:
        if e.alive and e.floor == agent.floor:
            ed = abs(e.x - agent.x) + abs(e.y - agent.y)
            if ed < min_enemy_dist:
                min_enemy_dist = ed
            ox, oy = e.x - agent.x + vr, e.y - agent.y + vr
            if 0 <= ox < vs and 0 <= oy < vs:
                ch = ETYPE_CH.get(e.etype, 10)
                obs[ch][oy][ox] = 1.0

    for bx, by, owner in bpos:
        ox, oy = bx - agent.x + vr, by - agent.y + vr
        if 0 <= ox < vs and 0 <= oy < vs:
            obs[13][oy][ox] = 1.0
    for lx, ly in lasers:
        ox, oy = lx - agent.x + vr, ly - agent.y + vr
        if 0 <= ox < vs and 0 <= oy < vs:
            obs[14][oy][ox] = 1.0

    gx, gy, gf = target_3d
    if gf == agent.floor:
        ref_x, ref_y = gx, gy
    else:
        target_stair = Tile.STAIR_UP if gf > agent.floor else Tile.STAIR_DOWN
        best_d, ref_x, ref_y = float("inf"), agent.x, agent.y
        for sy in range(h):
            for sx in range(w):
                if int(grid[sy][sx]) == int(target_stair):
                    d = abs(sx - agent.x) + abs(sy - agent.y)
                    if d < best_d:
                        best_d, ref_x, ref_y = d, sx, sy

    local_visited_density = float(obs[15].mean() * 2.0 - 1.0)
    enemy_prox = (
        -1.0
        if min_enemy_dist == float("inf")
        else (1.0 - min_enemy_dist / max(1, h + w)) * 2.0 - 1.0
    )
    time_rem = (1.0 - agent.steps / max(1, max_steps)) * 2.0 - 1.0
    cur_tile = int(grid[agent.y][agent.x])
    on_stair = (
        1.0
        if cur_tile in (Tile.STAIR_UP, Tile.STAIR_DOWN)
        else -1.0
    )
    floor_ratio = (
        (agent.floor / max(1, nf - 1) if nf > 1 else 0.0) * 2.0 - 1.0
    )
    key_ratio = (
        (agent.keys / max(1, agent.keys, 1)) * 2.0 - 1.0
    )

    vec = np.array(
        [
            (agent.hp / max(1, agent.max_hp)) * 2.0 - 1.0,
            (agent.ammo / max(1, agent.max_ammo)) * 2.0 - 1.0,
            (agent.stamina / max(1, agent.max_stamina)) * 2.0 - 1.0,
            np.clip(key_ratio, -1.0, 1.0),
            np.clip((agent.noise_level / 10.0) * 2.0 - 1.0, -1.0, 1.0),
            np.clip(floor_ratio, -1.0, 1.0),
            float(np.sign(ref_x - agent.x)),
            float(np.sign(ref_y - agent.y)),
            float(np.sign(gf - agent.floor)),
            float(agent.last_dir[0]),
            float(agent.last_dir[1]),
            np.clip(time_rem, -1.0, 1.0),
            np.clip(local_visited_density, -1.0, 1.0),
            on_stair,
            np.clip(enemy_prox, -1.0, 1.0),
        ],
        dtype=np.float32,
    )
    return obs, vec


In [ ]:
# --- Cell 8: MazeEnv ---
class MazeEnv(gym.Env):

    metadata = {"render_modes": ["rgb_array"]}

    REWARD = {
        "step": -0.02,
        "wall": -0.12,
        "goal": 25.0,
        "key": 2.5,
        "door": 3.5,
        "door_lk": -0.15,
        "kill": 1.25,
        "dmg": -2.0,
        "death": -18.0,
        "stairs": 0.75,
        "explore": 0.05,
        "trap": -2.5,
        "hp": 0.75,
        "ammo": 0.50,
        "approach": 0.10,
        "shoot": -0.04,
    }

    def __init__(self, config=None, render_mode="rgb_array"):
        super().__init__()
        self.cfg = config or {}
        self.render_mode = render_mode
        self.vr = self.cfg.get("view_radius", 4)
        vs = 2 * self.vr + 1
        self.observation_space = spaces.Dict(
            {
                "visual": spaces.Box(0, 1, (N_CHANNELS, vs, vs), np.float32),
                "vector": spaces.Box(-1, 1, (N_VEC,), np.float32),
            }
        )
        self.action_space = spaces.Discrete(10)
        self.R = copy.deepcopy(self.REWARD)
        self.R.update(self.cfg.get("rewards", {}))
        self.max_steps = self.cfg.get("max_steps", 200)
        self.floors = None
        self.agent = None
        self.enemies = []
        self.goal = (0, 0, 0)
        self.done = False
        self.win = False
        self.total_reward = 0
        self._lasers = []
        self._cached_dist = 999
        self._cache_key_state = None
        self.max_theoretical_reward = 0.0

    def _estimate_reward_upper_bound(self, map_data):
        """Ước lượng upper bound reward có thể lấy được trên map này.

        Bỏ qua penalty bước đi / trap / đạn, chỉ cộng:
        - reward goal
        - reward nhặt key / máu / đạn
        - reward stairs (mỗi ô cầu thang)
        - reward explore cho mọi ô có thể ghé qua (bfs không chặn cửa)
        """
        floors = map_data["floors"]
        sx, sy, sf = map_data["start"]
        goal = map_data["goal"]
        pmg = ProceduralMapGenerator()
        reachable = pmg._bfs_reachable_set(floors, (sx, sy, sf), set())
        tiles = set(reachable)
        R = self.R
        val = 0.0
        for (x, y, f) in tiles:
            t = int(floors[f][y][x])
            if t == int(Tile.KEY):
                val += R.get("key", 0.0)
            elif t == int(Tile.HEALTH_PACK):
                val += R.get("hp", 0.0)
            elif t == int(Tile.AMMO_PACK):
                val += R.get("ammo", 0.0)
            elif t in (int(Tile.STAIR_UP), int(Tile.STAIR_DOWN)):
                val += R.get("stairs", 0.0)
            elif t == int(Tile.GOAL):
                val += R.get("goal", 0.0)
        val += R.get("explore", 0.0) * len(tiles)
        # đảm bảo goal nếu reachable thì đã tính
        if goal not in tiles:
            gx, gy, gf = goal
            if 0 <= gf < len(floors):
                if 0 <= gy < floors[gf].shape[0] and 0 <= gx < floors[gf].shape[1]:
                    if int(floors[gf][gy][gx]) == int(Tile.GOAL):
                        val += R.get("goal", 0.0)
        return float(val)

    def _compute_distance(self):
        """BFS distance from agent to dynamic goal. Called sparingly."""
        dg = self._dynamic_goal()
        gx, gy, gf = dg
        ax, ay, af = self.agent.x, self.agent.y, self.agent.floor
        if ax == gx and ay == gy and af == gf:
            return 0
        h, w = self.floors[0].shape
        vis = {(ax, ay, af)}
        q = deque([(ax, ay, af, 0)])
        while q:
            cx, cy, cf, d = q.popleft()
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx, ny = cx + dx, cy + dy
                if 0 <= nx < w and 0 <= ny < h and (nx, ny, cf) not in vis:
                    t = int(self.floors[cf][ny][nx])
                    if t != int(Tile.WALL) and not (
                        t == int(Tile.DOOR) and self.agent.keys == 0
                    ):
                        if nx == gx and ny == gy and cf == gf:
                            return d + 1
                        vis.add((nx, ny, cf))
                        q.append((nx, ny, cf, d + 1))
            ct = int(self.floors[cf][cy][cx])
            if ct == int(Tile.STAIR_UP) and cf < len(self.floors) - 1:
                if (cx, cy, cf + 1) not in vis:
                    if cx == gx and cy == gy and cf + 1 == gf:
                        return d + 1
                    vis.add((cx, cy, cf + 1))
                    q.append((cx, cy, cf + 1, d + 1))
            elif ct == int(Tile.STAIR_DOWN) and cf > 0:
                if (cx, cy, cf - 1) not in vis:
                    if cx == gx and cy == gy and cf - 1 == gf:
                        return d + 1
                    vis.add((cx, cy, cf - 1))
                    q.append((cx, cy, cf - 1, d + 1))
        return 999

    def _key_door_state(self):
        """Hashable representation of key/door state for cache invalidation."""
        return (self.agent.keys, self.agent.x, self.agent.y, self.agent.floor)

    def _update_distance_cache(self, force=False):
        new_state = self._key_door_state()
        if force or new_state != self._cache_key_state:
            self._cached_dist = self._compute_distance()
            self._cache_key_state = new_state

    def _dynamic_goal(self):
        if self.agent.keys == 0:
            bd, bp = float("inf"), None
            for f, grid in enumerate(self.floors):
                h, w = grid.shape
                for sy in range(h):
                    for sx in range(w):
                        if grid[sy][sx] == Tile.KEY:
                            d = (
                                abs(sx - self.agent.x)
                                + abs(sy - self.agent.y)
                                + abs(f - self.agent.floor) * 20
                            )
                            if d < bd:
                                bd, bp = d, (sx, sy, f)
            if bp:
                return bp
        return self.goal

    def reset(self, seed=None, options=None):
        if options and "map_data" in options:
            md = options["map_data"]
        else:
            mg = ProceduralMapGenerator(seed)
            md = mg.generate(self.cfg)
        # estimate trước khi copy floors (dùng cho phân tích / baseline)
        self.max_theoretical_reward = self._estimate_reward_upper_bound(md)
        self.floors = [g.copy() for g in md["floors"]]
        self.nf = md["n_floors"]
        self.mw, self.mh = md["width"], md["height"]
        sx, sy, sf = md["start"]
        self.goal = md["goal"]

        hp = self.cfg.get("agent_hp", 3)
        ammo = self.cfg.get("agent_ammo", 3)
        stam = self.cfg.get("agent_stamina", 3)
        self.agent = AgentState(sx, sy, sf, hp, ammo, stam)
        self.agent.max_ammo = self.cfg.get("agent_max_ammo", 5)

        self.enemies = []
        for ec in self.cfg.get("enemies", []):
            ef = ec.get("floor", 0)
            g = self.floors[ef]
            ft = [
                (x, y)
                for y in range(self.mh)
                for x in range(self.mw)
                if g[y][x] == Tile.FLOOR and (x, y) != (sx, sy)
            ]
            ex, ey = random.choice(ft) if ft else (1, 1)
            et = ec.get("type", "patrol")
            if et == "patrol":
                wps = [(ex, ey)]
                for _ in range(3):
                    wps.append(
                        (
                            max(1, min(self.mw - 2, ex + random.randint(-4, 4))),
                            max(1, min(self.mh - 2, ey + random.randint(-4, 4))),
                        )
                    )
                self.enemies.append(PatrolEnemy(ex, ey, ef, wps))
            elif et == "chaser":
                self.enemies.append(ChaserEnemy(ex, ey, ef))
            elif et == "sniper":
                self.enemies.append(SniperEnemy(ex, ey, ef))

        self.ballistics = BallisticsEngine()
        self.stealth = StealthSystem()
        self.done = False
        self.win = False
        self.total_reward = 0
        self.agent.steps = 0
        self.agent.visited = {(sx, sy, sf)}
        self._lasers = []
        self._cache_key_state = None
        self._update_distance_cache(force=True)

        return self._obs(), self._info()

    def step(self, action):
        if self.done:
            return self._obs(), 0.0, True, False, self._info()

        dist_old = self._cached_dist
        self.agent.old_x, self.agent.old_y = self.agent.x, self.agent.y

        a = int(action)
        self.agent.steps += 1
        r = self.R["step"]
        g = self.floors[self.agent.floor]
        self.stealth.calc(a, self.agent)
        key_door_changed = False

        if a in (0, 1, 2, 3):
            dx, dy = DIR_DELTA[a]
            self.agent.last_dir = (dx, dy)
            mr, kdc = self._try_move(self.agent.x + dx, self.agent.y + dy)
            r += mr
            key_door_changed = kdc
        elif 4 <= a <= 7:
            dx, dy = DIR_DELTA[a - 4]
            self.agent.last_dir = (dx, dy)
            if self.agent.use_ammo():
                r += self.R["shoot"]
                self.ballistics.fire(
                    self.agent.x,
                    self.agent.y,
                    self.agent.floor,
                    dx,
                    dy,
                    "agent",
                    self.cfg.get("ricochet", 0),
                )
            else:
                r += self.R["wall"]
        elif a == 8:
            if self.agent.use_stamina():
                dx, dy = self.agent.last_dir
                for _ in range(2):
                    nx, ny = self.agent.x + dx, self.agent.y + dy
                    mr, kdc = self._try_move(nx, ny)
                    r += mr
                    key_door_changed = key_door_changed or kdc
                    # nếu bị chặn (tường/cửa không mở) thì dừng dash
                    if (self.agent.x, self.agent.y) != (nx, ny):
                        break
            else:
                r += self.R["wall"]
        elif a == 9:
            t = g[self.agent.y][self.agent.x]
            if t == Tile.STAIR_UP and self.agent.floor < self.nf - 1:
                self.agent.floor += 1
                pk2 = (self.agent.x, self.agent.y, self.agent.floor)
                if pk2 not in self.agent.visited:
                    r += self.R["stairs"]
                key_door_changed = True
            elif t == Tile.STAIR_DOWN and self.agent.floor > 0:
                self.agent.floor -= 1
                pk2 = (self.agent.x, self.agent.y, self.agent.floor)
                if pk2 not in self.agent.visited:
                    r += self.R["stairs"]
                key_door_changed = True
            else:
                r += self.R["wall"]

        pk = (self.agent.x, self.agent.y, self.agent.floor)
        if pk not in self.agent.visited:
            self.agent.visited.add(pk)
            r += self.R["explore"]

        # Approach reward: use cached distance, only recompute when needed
        self._update_distance_cache(force=key_door_changed)
        dist_new = self._cached_dist
        if self.R.get("approach", 0) > 0:
            r += self.R["approach"] * np.clip(dist_old - dist_new, -1, 1)

        for ev, _ in self.ballistics.update(g, self.enemies, self.agent):
            if ev == "kill":
                r += self.R["kill"]
            elif ev == "agent_hit":
                self.agent.damage(1)
                r += self.R["dmg"]

        self._lasers = []
        npos = self.stealth.npos
        for e in self.enemies:
            eg = self.floors[e.floor] if e.floor < self.nf else g
            if isinstance(e, SniperEnemy):
                proj = e.update(
                    eg,
                    (self.agent.x, self.agent.y),
                    self.agent.floor,
                    self.agent.noise_level,
                    npos,
                )
                if proj:
                    self.ballistics.add(proj)
                self._lasers.extend(e.laser_path)
            else:
                e.update(
                    eg,
                    (self.agent.x, self.agent.y),
                    self.agent.floor,
                    self.agent.noise_level,
                    npos,
                )

        for e in self.enemies:
            if (
                e.alive
                and e.floor == self.agent.floor
                and abs(e.x - self.agent.x) + abs(e.y - self.agent.y) <= 1
                and hasattr(e, "state")
                and e.state in ("investigate", "chase")
            ):
                self.agent.damage(1)
                r += self.R["dmg"]

        if self.agent.steps % 5 == 0:
            self.agent.regen_stamina()
        self.stealth.decay(self.agent)

        term = False
        gx, gy, gf = self.goal
        if not self.agent.alive:
            r += self.R["death"]
            term = True
        if (
            self.agent.x == gx
            and self.agent.y == gy
            and self.agent.floor == gf
        ):
            r += self.R["goal"]
            term = True
            self.win = True

        trunc = self.agent.steps >= self.max_steps
        self.done = term or trunc
        self.total_reward += r
        return self._obs(), float(r), term, trunc, self._info()

    def _try_move(self, nx, ny):
        """Returns (reward, key_door_changed)."""
        g = self.floors[self.agent.floor]
        h, w = g.shape
        if not (0 <= nx < w and 0 <= ny < h):
            return self.R["wall"], False
        t = int(g[ny][nx])
        if t == Tile.WALL:
            return self.R["wall"], False
        if t == Tile.DOOR:
            if self.agent.keys > 0:
                self.agent.keys -= 1
                g[ny][nx] = Tile.FLOOR
                self.agent.x, self.agent.y = nx, ny
                return self.R["door"], True
            return self.R["door_lk"], False
        self.agent.x, self.agent.y = nx, ny
        rv = 0.0
        kdc = False
        if t == Tile.TRAP:
            self.agent.damage(1)
            rv += self.R["trap"]
        elif t == Tile.KEY:
            self.agent.keys += 1
            g[ny][nx] = Tile.FLOOR
            rv += self.R["key"]
            kdc = True
        elif t == Tile.HEALTH_PACK:
            self.agent.heal(1)
            g[ny][nx] = Tile.FLOOR
            rv += self.R["hp"]
        elif t == Tile.AMMO_PACK:
            self.agent.add_ammo(2)
            g[ny][nx] = Tile.FLOOR
            rv += self.R["ammo"]
        return rv, kdc

    def _walk(self, nx, ny):
        g = self.floors[self.agent.floor]
        h, w = g.shape
        if not (0 <= nx < w and 0 <= ny < h):
            return False
        return int(g[ny][nx]) not in (Tile.WALL, Tile.DOOR)

    def _obs(self):
        g = self.floors[self.agent.floor]
        bp = self.ballistics.positions(self.agent.floor)
        dg = self._dynamic_goal()
        v, vec = _build_obs(
            g, self.agent, self.enemies, bp, self._lasers,
            self.nf, dg, self.max_steps, self.vr,
        )
        return {"visual": v, "vector": vec}

    def _info(self):
        return {
            "win": self.win,
            "steps": self.agent.steps,
            "hp": self.agent.hp,
            "ammo": self.agent.ammo,
            "floor": self.agent.floor,
            "keys": self.agent.keys,
            "total_reward": self.total_reward,
            "enemies_alive": sum(1 for e in self.enemies if e.alive),
            "explored": len(self.agent.visited),
            "max_theoretical_reward": getattr(
                self, "max_theoretical_reward", None
            ),
        }


### Bắt đầu Phần B

Train

In [ ]:
# --- B0: predict_first_step / predict_next_step ---
import numpy as np


def predict_first_step(model, obs, deterministic=True):
    action, _ = model.predict(obs, deterministic=deterministic)
    return int(action), None


def predict_next_step(model, obs, lstm_state, deterministic=True):
    action, _ = model.predict(obs, deterministic=deterministic)
    return int(action), None


In [ ]:
# --- B1: Import huấn luyện; MazeEnv & CURRICULUM đã định nghĩa ở Phần A ---
import argparse
import copy
import os
import time

import numpy as np
import torch
import torch.nn as nn
from gymnasium import spaces
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback


### Quan trọng:
Chỉ sửa **một** biến `MFM_ROOT` ở ô Python ngay dưới — mọi checkpoint `.zip`, `eval_metrics.json`, GIF và thống kê train đều dùng **cùng** thư mục đó.

In [ ]:
import os

# --- Chỉ sửa dòng MFM_ROOT: train, eval, phân tích, GIF đều dùng thư mục này ---
MFM_ROOT = "/Users/kotori/Documents/test_train"

_root = os.path.abspath(os.path.expanduser(str(MFM_ROOT).strip()))
os.environ["MFM_OUTPUT_DIR"] = _root
os.environ["MFM_ANALYSIS_DIR"] = _root
print("MFM_ROOT (OUTPUT_DIR + INPUT_DIR phân tích):", _root)


In [ ]:
# --- B2: Phần cứng, thư mục output, CURRICULUM_PLAN ---
import os

DEVICE = os.environ.get(
    "MFM_DEVICE",
    "cuda" if __import__("torch").cuda.is_available() else "cpu",
)
N_ENVS = int(os.environ.get("MFM_N_ENVS", "4"))

_base = (os.environ.get("MFM_OUTPUT_DIR") or os.environ.get("MFM_ANALYSIS_DIR") or "").strip()
if not _base:
    raise RuntimeError(
        "Chưa cấu thư mục làm việc: chạy ô MFM_ROOT phía trên (hoặc export MFM_OUTPUT_DIR)."
    )
OUTPUT_DIR = os.path.abspath(os.path.expanduser(_base))
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR (file .zip):", OUTPUT_DIR)

CURRICULUM_PLAN = [
    {"level": 1, "timesteps": 100_000, "mastery": 0.70},
    {"level": 2, "timesteps": 150_000, "mastery": 0.60},
    {"level": 3, "timesteps": 200_000, "mastery": 0.50},
    {"level": 4, "timesteps": 300_000, "mastery": 0.40},
    {"level": 5, "timesteps": 300_000, "mastery": 0.35},
    {"level": 6, "timesteps": 400_000, "mastery": 0.30},
    {"level": 7, "timesteps": 500_000, "mastery": 0.25},
    {"level": 8, "timesteps": 600_000, "mastery": 0.20},
    {"level": 9, "timesteps": 800_000, "mastery": 0.15},
]


In [ ]:
# --- B3: Feature extractors (CNN + vector) ---

class SmallDictExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Dict, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        n_ch = observation_space["visual"].shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_ch, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        vec_dim = observation_space["vector"].shape[0]
        self.vec_net = nn.Sequential(
            nn.Linear(vec_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.fusion = nn.Sequential(nn.Linear(64 + 64, features_dim), nn.ReLU())

    def forward(self, obs):
        vis = self.cnn(obs["visual"])
        vec = self.vec_net(obs["vector"])
        return self.fusion(torch.cat([vis, vec], dim=1))


In [ ]:
# --- B4: make_vec_env, LogCallback, evaluate ---

import time
import numpy as np


def make_env(level_config, seed=None):
    def _init():
        env = MazeEnv(config=copy.deepcopy(level_config))
        if seed is not None:
            env.reset(seed=seed)
        return env

    return _init


def make_vec_env(level_config, n_envs=N_ENVS, normalize=True):
    envs = DummyVecEnv([make_env(level_config, seed=i * 1000) for i in range(n_envs)])
    if normalize:
        envs = VecNormalize(
            envs,
            norm_obs=False,
            norm_reward=True,
            clip_reward=10.0,
            gamma=0.99,
        )
    return envs


class LogCallback(BaseCallback):
    """Ghi lại metrics train theo episode và theo mốc timestep (đủ để vẽ curve)."""

    def __init__(self, log_interval=10000, verbose=0, phase="train"):
        super().__init__(verbose)
        self.log_interval = log_interval
        self.phase = phase

        # episode-level buffers (dùng để tính rolling mean ở log point)
        self.ep_rewards: list[float] = []
        self.ep_wins: list[float] = []
        self.ep_regrets: list[float] = []
        self.ep_lens: list[int] = []

        # time-series points (nhỏ để lưu JSON)
        self.train_points: list[dict] = []

        self._running_rewards = None
        self._running_lens = None
        self._t0 = None

    def _on_training_start(self):
        self._running_rewards = np.zeros(self.training_env.num_envs, dtype=np.float64)
        self._running_lens = np.zeros(self.training_env.num_envs, dtype=np.int64)
        self._t0 = time.time()

    def _on_step(self):
        rewards = np.asarray(self.locals["rewards"], dtype=np.float64)
        dones = np.asarray(self.locals["dones"], dtype=bool)
        infos = self.locals.get("infos", [])

        self._running_rewards += rewards
        self._running_lens += 1

        for i, done in enumerate(dones):
            if not done:
                continue

            r = float(self._running_rewards[i])
            L = int(self._running_lens[i])
            self._running_rewards[i] = 0.0
            self._running_lens[i] = 0

            info = infos[i] if i < len(infos) else {}
            win = bool(info.get("win", False))

            max_theoretical = info.get("max_theoretical_reward", None)
            if max_theoretical is None:
                regret = float(np.nan)
            else:
                regret = float(max_theoretical) - r

            self.ep_rewards.append(r)
            self.ep_lens.append(L)
            self.ep_wins.append(1.0 if win else 0.0)
            self.ep_regrets.append(regret)

        if self.n_calls % self.log_interval == 0 and self.ep_rewards:
            recent_r = np.asarray(self.ep_rewards[-100:], dtype=np.float64)
            recent_w = np.asarray(self.ep_wins[-100:], dtype=np.float64)
            recent_regrets = np.asarray(self.ep_regrets[-100:], dtype=np.float64)
            recent_lens = np.asarray(self.ep_lens[-100:], dtype=np.float64)

            mean_r = float(np.mean(recent_r))
            win_rate = float(np.mean(recent_w)) if recent_w.size else 0.0
            mean_regret = float(np.nanmean(recent_regrets))
            mean_len = float(np.mean(recent_lens))

            wall_s = float(time.time() - self._t0) if self._t0 is not None else 0.0

            self.train_points.append(
                {
                    "phase": self.phase,
                    "timestep": int(self.num_timesteps),
                    "wall_s": wall_s,
                    "mean_reward_recent": mean_r,
                    "win_rate_recent": win_rate,
                    "mean_regret_recent": mean_regret,
                    "mean_ep_len_recent": mean_len,
                }
            )

            # giữ hành vi print cũ (nhưng thêm regret)
            print(
                f"  step={self.num_timesteps:>8d} | mean_r={mean_r:>7.2f} | mean_regret={mean_regret:>7.2f} | win={win_rate:>6.2%}"
            )

        return True


def evaluate(model, level_config, n_episodes=30):
    rewards, wins = [], []
    for seed in range(9000, 9000 + n_episodes):
        env = MazeEnv(config=copy.deepcopy(level_config))
        obs, _ = env.reset(seed=seed)
        done = False
        total_r = 0.0
        lstm_state = None
        first = True
        while not done:
            if first:
                action, lstm_state = predict_first_step(model, obs, deterministic=True)
                first = False
            else:
                action, lstm_state = predict_next_step(model, obs, lstm_state, deterministic=True)
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            done = term or trunc
        rewards.append(total_r)
        wins.append(1.0 if info.get("win", False) else 0.0)
    return float(np.mean(rewards)), float(np.mean(wins))


### Thuật toán 


In [ ]:
# --- B5: build_ppo / build_a2c / build_dqn ---

def build_ppo(env, *, gamma=0.99, ent_coef=0.01, features_dim=128, net_arch=None):
    if net_arch is None:
        net_arch = dict(pi=[128, 64], vf=[128, 64])
    extractor = SmallDictExtractor
    policy_kwargs = dict(
        features_extractor_class=extractor,
        features_extractor_kwargs=dict(features_dim=features_dim),
        net_arch=net_arch,
        activation_fn=nn.ReLU,
    )
    return PPO(
        "MultiInputPolicy", env,
        learning_rate=3e-4,
        n_steps=32,
        batch_size=16,
        n_epochs=4,
        gamma=gamma,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=ent_coef,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        device=DEVICE,
        verbose=0,
    )


def build_a2c(
    env,
    *,
    learning_rate=7e-4,
    n_steps=16,
    ent_coef=0.01,
    gae_lambda=0.95,
    vf_coef=0.5,
):
    """A2C baseline / biến thể — cùng kiến trúc CNN+vector (SmallDictExtractor)."""
    policy_kwargs = dict(
        features_extractor_class=SmallDictExtractor,
        features_extractor_kwargs=dict(features_dim=128),
        net_arch=dict(pi=[128, 64], vf=[128, 64]),
        activation_fn=nn.ReLU,
    )
    return A2C(
        "MultiInputPolicy", env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        gamma=0.99,
        gae_lambda=gae_lambda,
        ent_coef=ent_coef,
        vf_coef=vf_coef,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        device=DEVICE,
        verbose=0,
    )


def build_dqn(env):
    """DQN off-policy; cùng extractor để so sánh với on-policy."""
    policy_kwargs = dict(
        features_extractor_class=SmallDictExtractor,
        features_extractor_kwargs=dict(features_dim=128),
        net_arch=[128, 64],
        activation_fn=nn.ReLU,
    )
    return DQN(
        "MultiInputPolicy", env,
        learning_rate=1e-4,
        buffer_size=200_000,
        learning_starts=20_000,
        batch_size=32,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=2000,
        exploration_fraction=0.15,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.05,
        gamma=0.99,
        max_grad_norm=10.0,
        policy_kwargs=policy_kwargs,
        device=DEVICE,
        verbose=0,
    )


In [ ]:
# --- B6: ExperimentSpec + đăng ký EXPERIMENTS (clean registry) ---
from dataclasses import dataclass
from typing import Any, Callable

# Chú thích tham số chung (xem build_* trong file):
#   gamma         — hệ số chiết khấu phần thưởng tương lai (0.99 ≈ ưu tiên dài hạn).
#   gae_lambda    — GAE: 0 → TD một bước; 1 → Monte Carlo; 0.95 cân bằng bias/variance.
#   ent_coef      — trọng số entropy: thúc đẩy thăm dò; 0 = không khuyến khích nhiễu.
#   vf_coef       — trọng số loss vô hình; cân đối policy vs value.
#   lr            — tốc độ học; A2C mặc định SB3 dùng RMSprop.
#   n_steps       — số bước rollout / env trước mỗi update; lớn hơn → gradient ổn định hơn nhưng chậm phản ứng.
#   PPO: n_epochs, batch_size, clip_range — cập nhật nhiều lần / clip surrogate.
#   DQN: buffer_size, replay, epsilon-greedy decay, target_net — học off-policy.


@dataclass(frozen=True)
class ExperimentSpec:
    """Một thí nghiệm: nhãn hiển thị + factory tạo model SB3 từ vec_env."""

    name: str
    build: Callable[[Any], Any]


# Đăng ký thí nghiệm — mỗi key map tới một cấu hình (dễ đọc, dễ mở rộng).
EXPERIMENTS: dict[str, ExperimentSpec] = {
    # ── So sánh thuật toán ─────────────────────────────────
    # exp1_ppo: lr=3e-4, n_steps=256, batch=128, n_epochs=4, clip_range=0.2, gae_lambda=0.95, ent=0.01
    "exp1_ppo": ExperimentSpec(
        name="PPO (on-policy, baseline so sánh)",
        build=lambda env: build_ppo(env),
    ),
    # exp6_dqn: lr=1e-4, buffer=200k, learning_starts=20k, ...
    "exp6_dqn": ExperimentSpec(
        name="DQN (off-policy, replay buffer)",
        build=build_dqn,
    ),
    # ── 4× A2C (cùng CNN+vector 128-d, net pi/vf [128,64]) ───
    "exp2_a2c": ExperimentSpec(
        name="A2C baseline (lr=7e-4, n_steps=16, ent=0.01)",
        build=lambda env: build_a2c(env),
    ),
    "exp3_a2c_no_ent": ExperimentSpec(
        name="A2C ablation: ent_coef=0 (entropy — chứng minh khám phá quan trọng)",
        build=lambda env: build_a2c(env, ent_coef=0.0),
    ),
    "exp4_a2c_lr_low": ExperimentSpec(
        name="A2C tinh chỉnh: learning_rate=3e-4 (ổn định hơn baseline)",
        build=lambda env: build_a2c(env, learning_rate=3e-4),
    ),
    "exp5_a2c_rollout32": ExperimentSpec(
        name="A2C tinh chỉnh: n_steps=32 (rollout dài, gradient ít noisy)",
        build=lambda env: build_a2c(env, n_steps=32),
    ),
}



In [ ]:
# --- B7: train_one ---

import json


def train_one(exp_id):
    spec = EXPERIMENTS[exp_id]
    print("\n" + "=" * 60)
    print(spec.name)
    print("=" * 60)

    exp_started = time.time()

    history = {
        "exp_id": exp_id,
        "exp_name": spec.name,
        "device": DEVICE if "DEVICE" in globals() else None,
        "n_envs": int(N_ENVS) if "N_ENVS" in globals() else None,
        "output_dir": OUTPUT_DIR if "OUTPUT_DIR" in globals() else None,
        "curriculum_plan": CURRICULUM_PLAN,
        "stages": [],
        "train_points_all": [],
        "timing": {"total_wall_s": None},
        "checkpoints": {},
    }

    first_cfg = copy.deepcopy(CURRICULUM[CURRICULUM_PLAN[0]["level"]])
    vec_env = make_vec_env(first_cfg)
    model = spec.build(vec_env)

    n_params = sum(p.numel() for p in model.policy.parameters())
    print(f"Params: {n_params:,}")
    history["n_params"] = int(n_params)

    for stage_idx, stage in enumerate(CURRICULUM_PLAN):
        level = stage["level"]
        timesteps = stage["timesteps"]
        mastery = stage["mastery"]
        level_cfg = copy.deepcopy(CURRICULUM[level])
        level_name = level_cfg.get("name", f"L{level}")

        stage_record = {
            "stage_idx": int(stage_idx),
            "level": int(level),
            "level_name": str(level_name),
            "timesteps_requested": int(timesteps),
            "mastery_win_rate": float(mastery),
            "train": [],
            "eval": {},
            "retry": None,
            "checkpoint_path": None,
        }

        print(f"\n--- Level {level} {level_name} | {timesteps:,} steps | mastery {mastery:.0%} ---")

        vec_env = make_vec_env(level_cfg)
        model.set_env(vec_env)

        t0 = time.time()
        cb = LogCallback(log_interval=10000, verbose=0, phase=f"{exp_id}_level{level}")
        model.learn(
            total_timesteps=timesteps,
            callback=cb,
            reset_num_timesteps=False,
        )
        elapsed = time.time() - t0

        stage_record["train"].append(
            {
                "phase": cb.phase,
                "wall_s": float(elapsed),
                "steps": int(timesteps),
                "steps_per_s": float(timesteps) / float(elapsed) if elapsed > 0 else None,
                "train_points": cb.train_points,
            }
        )
        history["train_points_all"].extend(cb.train_points)

        mean_r, win_rate = evaluate(model, level_cfg, n_episodes=30)
        print(f"  Done {elapsed:.0f}s | reward={mean_r:.2f} | win={win_rate:.1%}")

        stage_record["eval"] = {
            "eval_episodes": 30,
            "reward_mean": float(mean_r),
            "win_rate_mean": float(win_rate),
        }

        ckpt = os.path.join(OUTPUT_DIR, f"{exp_id}_level_{level}.zip")
        model.save(ckpt)
        stage_record["checkpoint_path"] = ckpt

        if win_rate < mastery:
            extra = min(timesteps, 200_000)
            print(f"  Below mastery, +{extra:,} steps")

            t1 = time.time()
            cb2 = LogCallback(log_interval=10000, verbose=0, phase=f"{exp_id}_level{level}_retry")
            model.learn(
                total_timesteps=extra,
                callback=cb2,
                reset_num_timesteps=False,
            )
            elapsed2 = time.time() - t1

            mean_r2, wr2 = evaluate(model, level_cfg, n_episodes=30)
            print(f"  Retry: reward={mean_r2:.2f} | win={wr2:.1%}")

            model.save(ckpt)

            stage_record["retry"] = {
                "steps": int(extra),
                "wall_s": float(elapsed2),
                "steps_per_s": float(extra) / float(elapsed2) if elapsed2 > 0 else None,
                "eval_after_retry": {
                    "reward_mean": float(mean_r2),
                    "win_rate_mean": float(wr2),
                },
                "train_points": cb2.train_points,
            }

            history["train_points_all"].extend(cb2.train_points)

        history["stages"].append(stage_record)

    final_path = os.path.join(OUTPUT_DIR, f"{exp_id}_final.zip")
    model.save(final_path)
    history["checkpoints"]["final"] = final_path

    history["timing"]["total_wall_s"] = float(time.time() - exp_started)

    train_stats_path = os.path.join(OUTPUT_DIR, f"{exp_id}_train_stats.json")
    with open(train_stats_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    print(f"\nSaved: {final_path}")
    print(f"Saved stats JSON: {train_stats_path}")

    print("\nFinal eval:")
    for stage in CURRICULUM_PLAN:
        level = stage["level"]
        cfg = copy.deepcopy(CURRICULUM[level])
        mr, wr = evaluate(model, cfg, n_episodes=30)
        print(f"  L{level:>2d} {cfg['name']:<12s} | reward={mr:>7.2f} | win={wr:.1%}")

    return final_path


### B8 — Chạy train


In [ ]:
# --- B8a — PPO (exp1_ppo) ---
train_one("exp1_ppo")


In [ ]:
# --- B8b — DQN (exp6_dqn) ---
# train_one("exp6_dqn")


In [ ]:
# --- B8d — A2C gốc (exp2_a2c) ---
# train_one("exp2_a2c")


In [ ]:
# --- B8e — A2C không entropy (exp3_a2c_no_ent) ---
# train_one("exp3_a2c_no_ent")


In [ ]:
# --- B8f — A2C lr thấp (exp4_a2c_lr_low) ---
# train_one("exp4_a2c_lr_low")


In [ ]:
# --- B8g — A2C rollout 32 (exp5_a2c_rollout32) ---
# train_one("exp5_a2c_rollout32")


In [ ]:
# --- C1: Render utilities (theo multi_floor_maze_main) — chạy sau A8 ---
import io
import base64
from PIL import Image, ImageDraw, ImageFont


def _font(size=20):
    for p in (
        "Arial.ttf",
        "DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    ):
        try:
            return ImageFont.truetype(p, size)
        except OSError:
            pass
    return ImageFont.load_default()


def img_to_base64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


ACT_NAMES = [
    "UP", "DOWN", "LEFT", "RIGHT",
    "SHOOT_U", "SHOOT_D", "SHOOT_L", "SHOOT_R",
    "DASH", "STAIRS",
]

TILE_COLOR = {
    Tile.FLOOR: (250, 250, 250),
    Tile.WALL: (40, 40, 40),
    Tile.STAIR_UP: (180, 180, 255),
    Tile.STAIR_DOWN: (150, 150, 220),
    Tile.GOAL: (255, 255, 100),
    Tile.TRAP: (255, 80, 80),
    Tile.START: (180, 255, 180),
    Tile.DOOR: (180, 140, 100),
    Tile.KEY: (220, 180, 80),
    Tile.HEALTH_PACK: (255, 200, 200),
    Tile.AMMO_PACK: (240, 240, 150),
}
TILE_LABEL = {
    Tile.STAIR_UP: "SU",
    Tile.STAIR_DOWN: "SD",
    Tile.GOAL: "G",
    Tile.TRAP: "X",
    Tile.START: "S",
    Tile.DOOR: "D",
    Tile.KEY: "K",
    Tile.HEALTH_PACK: "+",
    Tile.AMMO_PACK: "A",
}


def render_env(env, cell_px=48):
    g = env.floors[env.agent.floor]
    h, w = g.shape
    img = Image.new("RGB", (w * cell_px, h * cell_px), (255, 255, 255))
    d = ImageDraw.Draw(img)
    ft = _font(int(cell_px * 0.6))
    fu = _font(int(cell_px * 0.45))
    for y in range(h):
        for x in range(w):
            t = int(g[y][x])
            x1, y1 = x * cell_px, y * cell_px
            c = TILE_COLOR.get(t, (250, 250, 250))
            d.rectangle(
                [x1, y1, x1 + cell_px - 2, y1 + cell_px - 2],
                fill=c,
                outline=(150, 150, 150),
            )
            lb = TILE_LABEL.get(t)
            if lb:
                d.text(
                    (x1 + cell_px / 2, y1 + cell_px / 2),
                    lb,
                    fill=(0, 0, 0),
                    font=ft,
                    anchor="mm",
                )
    EC = {
        "patrol": (200, 100, 200, "P"),
        "chaser": (255, 50, 50, "E"),
        "sniper": (255, 165, 0, "S"),
    }
    for e in env.enemies:
        if e.alive and e.floor == env.agent.floor:
            ec = EC.get(e.etype, (150, 150, 150, "?"))
            d.ellipse(
                [
                    e.x * cell_px + 4,
                    e.y * cell_px + 4,
                    e.x * cell_px + cell_px - 6,
                    e.y * cell_px + cell_px - 6,
                ],
                fill=ec[:3],
                outline=(0, 0, 0),
                width=2,
            )
            d.text(
                (e.x * cell_px + cell_px / 2, e.y * cell_px + cell_px / 2),
                ec[3],
                fill=(255, 255, 255),
                font=fu,
                anchor="mm",
            )
    bp = env.ballistics.positions(env.agent.floor)
    for bx, by, owner in bp:
        bc = (200, 200, 0) if owner == "agent" else (255, 0, 0)
        d.ellipse(
            [
                bx * cell_px + cell_px // 3,
                by * cell_px + cell_px // 3,
                bx * cell_px + 2 * cell_px // 3,
                by * cell_px + 2 * cell_px // 3,
            ],
            fill=bc,
        )
    ax, ay = env.agent.x * cell_px + 6, env.agent.y * cell_px + 6
    ac = (
        (255, 100, 100)
        if env.agent.hp <= 1
        else (100, 200, 100)
        if env.agent.keys > 0
        else (100, 150, 255)
    )
    d.ellipse(
        [ax, ay, ax + cell_px - 12, ay + cell_px - 12],
        fill=ac,
        outline=(0, 0, 0),
        width=2,
    )
    d.text(
        (env.agent.x * cell_px + cell_px / 2, env.agent.y * cell_px + cell_px / 2),
        "@",
        fill=(0, 0, 0),
        font=fu,
        anchor="mm",
    )
    return img


print("Render utilities & ACT_NAMES loaded")


In [ ]:
# --- C2: Khởi tạo game — chạy sau C1 (theo multi_floor_maze_main) ---
import copy

sel = z.select(
    "Curriculum Level",
    [
        ("1", "L1: Crawl"),
        ("2", "L2: Walk"),
        ("3", "L3: Trapper"),
        ("4", "L4: Locksmith"),
        ("5", "L5: Climber"),
        ("6", "L6: Scout"),
        ("7", "L7: Operative"),
        ("8", "L8: Elite"),
        ("9", "L9: Master"),
    ],
    "1",
)

clvl = int(sel) if sel else 1
cfg = copy.deepcopy(CURRICULUM[clvl])
env = MazeEnv(config=cfg)
obs, info = env.reset()

_saved_map_data = {
    "floors": [g.copy() for g in env.floors],
    "n_floors": env.nf,
    "width": env.mw,
    "height": env.mh,
    "start": (env.agent.x, env.agent.y, env.agent.floor),
    "goal": env.goal,
}


def update_ui():
    global obs
    z.z.angularBind("frame", img_to_base64(render_env(env)))
    z.z.angularBind("floor_info", f"Floor {env.agent.floor}/{env.nf - 1}")
    z.z.angularBind("steps_info", f"{env.agent.steps}/{env.max_steps}")
    z.z.angularBind("score", f"{env.total_reward:.1f}")
    z.z.angularBind("hp_info", f"{env.agent.hp}/{env.agent.max_hp}")
    z.z.angularBind("ammo_info", f"{env.agent.ammo}/{env.agent.max_ammo}")
    z.z.angularBind("stam_info", f"{env.agent.stamina}/{env.agent.max_stamina}")
    z.z.angularBind("noise_info", str(env.agent.noise_level))
    z.z.angularBind("keys_info", str(env.agent.keys))
    z.z.angularBind(
        "enemies_info",
        str(sum(1 for e in env.enemies if e.alive and e.floor == env.agent.floor)),
    )
    vec = obs["vector"]
    z.z.angularBind(
        "target_vec", f"[{vec[6]:.2f}, {vec[7]:.2f}, {vec[8]:.1f}]"
    )


z.z.angularBind("lvl", cfg["name"])
z.z.angularBind("status", "Ready")
update_ui()
print(
    f"Level: {cfg['name']} | Grid: {cfg['width']}x{cfg['height']} | Floors: {cfg['n_floors']}"
)


%angular
<style>
.ni{display:flex;gap:20px;font-family:Arial;background:#f5f5f5;padding:15px;border-radius:8px;outline:none;color:#333}
.ni:focus{box-shadow:0 0 0 3px #4CAF50}
.ni-map img{border:2px solid #333;border-radius:4px;image-rendering:pixelated}
.ni-p{width:240px;display:flex;flex-direction:column;gap:8px}
.ni-i{background:#fff;padding:10px;border-radius:6px;border:1px solid #ddd}
.ni-i div{margin:3px 0;display:flex;justify-content:space-between;font-size:12px}
.ni-i .l{color:#666}.ni-i .v{font-weight:bold}
.ni-s{background:#007;color:#fff;padding:12px;border-radius:6px;text-align:center}
.ni-s .n{font-size:24px;font-weight:bold}
.ni-c{background:#fff;padding:10px;border-radius:6px;border:1px solid #ddd}
.ni-mv{display:flex;flex-direction:column;align-items:center;gap:3px;margin-bottom:6px}
.ni-mv .r{display:flex;gap:3px}
.b{flex:1;padding:8px;cursor:pointer;border:1px solid #aaa;border-radius:4px;background:#fff;color:#333;font-size:12px;text-align:center;min-width:40px}
.b:hover{background:#eee}.b:active{background:#ddd}
.br{background:#ffe0e0;width:100%;margin-top:6px}
.br:hover{background:#fcc}.br:focus,.br:active{outline:none;border:1px solid #c99}
.ni-o{background:#fff;padding:8px;border-radius:6px;border:1px solid #ddd;font-size:10px;font-family:monospace}
.ni-o .t{font-weight:bold;margin-bottom:3px;font-size:11px;color:#007}
.ni-st{background:#fff;padding:8px;border-radius:6px;text-align:center;font-size:12px;border:1px solid #ddd;color:#333}
.sec{font-size:10px;color:#999;text-align:center;margin-top:3px;letter-spacing:1px}
.kh{display:flex;gap:8px;justify-content:center;font-size:9px;color:#888;margin-top:4px;flex-wrap:wrap}
.kh span{background:#f0f0f0;padding:2px 5px;border-radius:3px;border:1px solid #ccc;font-family:monospace}
.dg{color:#c00}.wr{color:#c80}.gd{color:#060}
</style>
<div class="ni" tabindex="0" ng-keydown="k=$event.key.toLowerCase(); k=='w'&&z.runParagraph('paragraph_1774168812353_375556951'); k=='s'&&z.runParagraph('paragraph_1774168812353_934015518'); k=='a'&&z.runParagraph('paragraph_1774168812353_188827817'); k=='d'&&z.runParagraph('paragraph_1774168812353_2088190052'); k=='e'&&z.runParagraph('paragraph_1774168812353_1488525513'); k=='q'&&z.runParagraph('paragraph_1774168812353_2052636316'); k=='i'&&z.runParagraph('paragraph_1774168812353_50104629'); k=='k'&&z.runParagraph('paragraph_1774168812353_1489520419'); k=='j'&&z.runParagraph('paragraph_1774168812353_1641914961'); k=='l'&&z.runParagraph('paragraph_1774168812353_1985651453'); k==' '&&z.runParagraph('paragraph_1774168812353_641176049'); k=='r'&&z.runParagraph('paragraph_1774168812353_1615363894');">
<div class="ni-map"><img ng-src="data:image/png;base64,{{frame}}"></div>
<div class="ni-p">
<div class="ni-i">
 <div><span class="l">Level</span><span class="v">{{lvl}}</span></div>
 <div><span class="l">Floor</span><span class="v">{{floor_info}}</span></div>
 <div><span class="l">Steps</span><span class="v">{{steps_info}}</span></div>
 <div><span class="l">HP</span><span class="v dg">{{hp_info}}</span></div>
 <div><span class="l">Ammo</span><span class="v wr">{{ammo_info}}</span></div>
 <div><span class="l">Stamina</span><span class="v gd">{{stam_info}}</span></div>
 <div><span class="l">Noise</span><span class="v">{{noise_info}}</span></div>
 <div><span class="l">Keys</span><span class="v gd">{{keys_info}}</span></div>
 <div><span class="l">Enemies</span><span class="v dg">{{enemies_info}}</span></div>
</div>
<div class="ni-s"><div style="font-size:11px">SCORE</div><div class="n">{{score}}</div></div>
<div class="ni-c">
 <div class="sec">MOVE (WASD)</div>
 <div class="ni-mv">
  <div class="r"><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_375556951')">W &uarr;</button></div>
  <div class="r"><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_188827817')">A &larr;</button><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_934015518')">S &darr;</button><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_2088190052')">D &rarr;</button></div>
 </div>
 <div class="sec">SHOOT (IJKL)</div>
 <div class="ni-mv">
  <div class="r"><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_50104629')">I &uarr;</button></div>
  <div class="r"><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_1641914961')">J &larr;</button><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_1489520419')">K &darr;</button><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_1985651453')">L &rarr;</button></div>
 </div>
 <div class="ni-mv"><div class="r"><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_2052636316')">Q Dash</button><button class="b" ng-click="z.runParagraph('paragraph_1774168812353_1488525513')">E Stairs</button></div></div>
 <div style="display:flex;gap:5px;width:100%;margin-top:6px">
 <button class="b br" style="margin-top:0" ng-click="z.runParagraph('paragraph_1774168812353_1615363894')">RETRY STAGE (R)</button>
 <button class="b br" style="margin-top:0" ng-click="z.runParagraph('paragraph_1774168812353_641176049')">NEW MAP (Space)</button>
</div>
</div>
<div class="ni-o"><div class="t">Target</div><div>{{target_vec}}</div></div>
<div class="ni-st">{{status}}</div>
<div class="kh"><span>W</span><span>A</span><span>S</span><span>D</span>move <span>I</span><span>J</span><span>K</span><span>L</span>shoot <span>Q</span>dash <span>E</span>stairs <span>R</span>retry <span>Space</span>new map</div>
</div>
</div>


# Các cell hành động


In [ ]:
def act(a):
    global obs
    if env.done:
        z.z.angularBind("status", "Game Over! Press NEW MAP")
        return
    old_floor = env.agent.floor
    obs, r, term, trunc, info = env.step(a)
    update_ui()
    if not env.agent.alive:
        z.z.angularBind("status", f"DEATH! Score: {env.total_reward:.0f}")
    elif info.get("win"):
        z.z.angularBind("status", f"WIN! Score: {env.total_reward:.0f}")
    elif trunc:
        z.z.angularBind("status", "Timeout!")
    elif env.agent.floor != old_floor:
        z.z.angularBind("status", f"Floor {env.agent.floor} ({r:+.1f})")
    else:
        z.z.angularBind("status", f"{ACT_NAMES[a]} ({r:+.2f})")


print("Action handler ready (10 actions)")


In [ ]:
# Di chuyển LÊN (W) | title: pu | MazeEnv Discrete action = 0
act(0)


In [ ]:
# Di chuyển XUỐNG (S) | title: pd | action = 1
act(1)


In [ ]:
# Di chuyển TRÁI (A) | title: pl | action = 2
act(2)


In [ ]:
# Di chuyển PHẢI (D) | title: pr | action = 3
act(3)


In [ ]:
# Bắn LÊN (I) | title: pshoot_u | action = 4
act(4)


In [ ]:
# Bắn XUỐNG (K) | title: pshoot_d | action = 5
act(5)


In [ ]:
# Bắn TRÁI (J) | title: pshoot_l | action = 6
act(6)


In [ ]:
# Bắn PHẢI (L) | title: pshoot_r | action = 7
act(7)


In [ ]:
# Dash (Q) | title: pdash | action = 8
act(8)


In [ ]:
# Cầu thang lên/xuống (E) | title: pstairs | action = 9
act(9)


In [ ]:

global obs, _saved_map_data
obs, _ = env.reset()
_saved_map_data = {
    "floors": [g.copy() for g in env.floors],
    "n_floors": env.nf,
    "width": env.mw,
    "height": env.mh,
    "start": (env.agent.x, env.agent.y, env.agent.floor),
    "goal": env.goal,
}
update_ui()
z.z.angularBind("status", "New Map!")


In [ ]:

global obs
obs, _ = env.reset(options={"map_data": _saved_map_data})
update_ui()
z.z.angularBind("status", "Restarted!")


### Phần D — GIF checkpoint
Tạo file gif theo config 


In [ ]:
# --- D-helpers: load + rollout + bind (chạy một lần sau B5/B6) ---
import base64
import copy
import os
import time
import json

import torch.nn as nn
from stable_baselines3 import A2C, DQN, PPO

def _gif_output_dir():
    for _k in ("MFM_OUTPUT_DIR", "MFM_ANALYSIS_DIR"):
        env_dir = os.environ.get(_k)
        if env_dir and str(env_dir).strip():
            return os.path.abspath(os.path.expanduser(str(env_dir).strip()))
    if "OUTPUT_DIR" in globals() and OUTPUT_DIR:
        return os.path.abspath(os.path.expanduser(str(OUTPUT_DIR)))
    return os.path.abspath(os.getcwd())

def load_model_for_exp(exp_id, path):
    dev = DEVICE if "DEVICE" in globals() else "cpu"
    cfg = copy.deepcopy(CURRICULUM[CURRICULUM_PLAN[0]["level"]])
    vec_env = make_vec_env(cfg, n_envs=1, normalize=True)
    pol_shared = dict(
        features_extractor_class=SmallDictExtractor,
        features_extractor_kwargs=dict(features_dim=128),
        net_arch=dict(pi=[128, 64], vf=[128, 64]),
        activation_fn=nn.ReLU,
        share_features_extractor=True,
    )
    if exp_id == "exp6_dqn":
        return DQN.load(path, env=vec_env, device=dev)
    if exp_id == "exp1_ppo":
        return PPO.load(path, env=vec_env, device=dev, custom_objects={"policy_kwargs": pol_shared})
    if exp_id in ("exp2_a2c", "exp3_a2c_no_ent", "exp4_a2c_lr_low", "exp5_a2c_rollout32"):
        return A2C.load(path, env=vec_env, device=dev, custom_objects={"policy_kwargs": pol_shared})
    raise ValueError("exp_id khong hop le: " + str(exp_id))


def update_gif_meta(fname, n_frames, win, elapsed):
    OUTPUT_DIR_GIF = _gif_output_dir()
    GIF_SUBDIR = os.path.join(OUTPUT_DIR_GIF, "gifs")
    meta_path = os.path.join(GIF_SUBDIR, "metadata.json")
    try:
        with open(meta_path, "r") as f:
            meta = json.load(f)
    except Exception:
        meta = {}
    meta[fname] = {"frames": n_frames, "win": win, "time": elapsed}
    with open(meta_path, "w") as f:
        json.dump(meta, f)

def record_episode_gif(model, level_cfg, seed, out_path, max_frames=800, frame_ms=300):
    cfg = copy.deepcopy(level_cfg)
    env_gif = MazeEnv(config=cfg)
    obs, _ = env_gif.reset(seed=seed)
    frames = [render_env(env_gif)]
    done, n, first, lstm_state, info = False, 0, True, None, {}
    while not done and n < max_frames:
        if first:
            action, lstm_state = predict_first_step(model, obs, deterministic=True)
            first = False
        else:
            action, lstm_state = predict_next_step(model, obs, lstm_state, deterministic=True)
        obs, r, term, trunc, info = env_gif.step(action)
        n += 1
        frames.append(render_env(env_gif))
        done = term or trunc
    if n >= max_frames:
        print("  WARNING: dat max_frames=%s" % max_frames)
    if not frames:
        return 0, False
    if len(frames) > 1:
        frames[0].save(out_path, format="GIF", save_all=True, append_images=frames[1:], duration=frame_ms, loop=0)
    else:
        frames[0].save(out_path, format="GIF")
    return len(frames), bool(info.get("win", False))

def run_gif_batch_custom(exp_id, level_cfg, base_seed):
    OUTPUT_DIR_GIF = _gif_output_dir()
    GIF_SUBDIR = os.path.join(OUTPUT_DIR_GIF, "gifs")
    os.makedirs(GIF_SUBDIR, exist_ok=True)
    level_name = level_cfg.get("name", "Custom")
    print("Sinh GIF cho: exp_id=%s | config=%s | seed=%s" % (exp_id, level_name, base_seed))
    
    for k in range(1, 10):
        ckpt = os.path.join(OUTPUT_DIR_GIF, "%s_level_%s.zip" % (exp_id, k))
        if not os.path.isfile(ckpt):
            print("  SKIP k=%s: khong co file checkpoint" % k)
            continue
        fname = "gif_%s_ckpt%s_seed%s.gif" % (exp_id, k, base_seed)
        out_path = os.path.join(GIF_SUBDIR, fname)
        t0 = time.time()
        try:
            model = load_model_for_exp(exp_id, ckpt)
        except Exception as e:
            print("  ERROR load k=%s: %s" % (k, e))
            continue
        try:
            n_frames, win = record_episode_gif(model, level_cfg, base_seed, out_path)
            elapsed = time.time() - t0
            update_gif_meta(fname, n_frames, win, elapsed)
            print("  [%s/9] %s | frames=%s win=%s (%.1fs)" % (k, fname, n_frames, win, elapsed))
        except Exception as e:
            print("  ERROR rollout k=%s: %s" % (k, e))
            continue

def load_recent_gifs_to_angular():
    OUTPUT_DIR_GIF = _gif_output_dir()
    GIF_SUBDIR = os.path.join(OUTPUT_DIR_GIF, "gifs")
    meta_path = os.path.join(GIF_SUBDIR, "metadata.json")
    try:
        with open(meta_path, "r") as f:
            meta = json.load(f)
    except Exception:
        meta = {}
    
    gifs = []
    if os.path.isdir(GIF_SUBDIR):
        files = [f for f in os.listdir(GIF_SUBDIR) if f.endswith(".gif")]

        files.sort(key=lambda x: os.path.getmtime(os.path.join(GIF_SUBDIR, x)), reverse=True)
        files = files 
        
        for fname in files:
            p = os.path.join(GIF_SUBDIR, fname)
            try:
                with open(p, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode("ascii")
            except Exception:
                continue
            
            m = meta.get(fname, {})
            win_str = ""
            if "win" in m:
                win_str = "WIN" if m["win"] else "LOSE"
            frames_str = "%s frames" % m.get("frames", "?") if "frames" in m else ""
            time_str = "%.1fs" % m.get("time", 0) if "time" in m else ""
            
            m_parts = [p for p in [win_str, frames_str, time_str] if p]
            m_str = " | ".join(m_parts)
            
            gifs.append({"name": fname, "b64": b64, "meta": m_str})
    
    z.z.angularBind("d_all_gifs", gifs)
    return len(gifs)

def d_init_angular():
    z.z.angularBind("d_gif_status", "Sẵn sàng — điền cấu hình ở D-generate.")
    load_recent_gifs_to_angular()

d_init_angular()
print("D-helpers OK.")



In [ ]:
# --- D-generate: sinh 1 GIF / 1 checkpoint (safe loader) ---
import copy
import os
import time

from stable_baselines3.common.save_util import load_from_zip_file


def _normalize_exp_id(exp_id):
    if isinstance(exp_id, (list, tuple)):
        found = None
        for x in exp_id:
            if isinstance(x, str) and x in EXPERIMENTS:
                found = x
                break
        if found is not None:
            exp_id = found
        elif len(exp_id) > 0:
            exp_id = exp_id[0]
        else:
            raise ValueError("exp_id rong")

    if not isinstance(exp_id, str):
        exp_id = str(exp_id)

    if exp_id not in EXPERIMENTS:
        raise ValueError("exp_id khong hop le: %r" % (exp_id,))

    return exp_id


def load_model_for_exp(exp_id, ckpt_path):
    exp_id = _normalize_exp_id(exp_id)
    dev = DEVICE if "DEVICE" in globals() else "cpu"

    first_level = 1
    if "CURRICULUM_PLAN" in globals() and CURRICULUM_PLAN:
        first_level = int(CURRICULUM_PLAN[0]["level"])

    cfg = copy.deepcopy(CURRICULUM[first_level])
    vec_env = make_vec_env(cfg, n_envs=1, normalize=True)

    # Dung model moi bang chinh factory luc train.
    model = EXPERIMENTS[exp_id].build(vec_env)

    # Chi nap weights policy, KHONG deserialize policy_kwargs trong checkpoint.
    # Cach nay tranh loi cloudpickle/Python version mismatch:
    #   code() takes at most 16 arguments (18 given)
    _, params, _ = load_from_zip_file(ckpt_path, load_data=False, device=dev)

    if "policy" not in params:
        raise KeyError("Checkpoint khong co state_dict 'policy': %s" % ckpt_path)

    model.policy.load_state_dict(params["policy"], strict=True)
    model.policy.set_training_mode(False)
    return model


EXP_KEYS = list(EXPERIMENTS.keys())
EXP_OPTIONS = [(k, EXPERIMENTS[k].name) for k in EXP_KEYS]

config_mode = z.select(
    "Config mode",
    [("curriculum", "CURRICULUM"), ("custom", "Tuỳ chỉnh")],
    "curriculum",
)

selected_exp = z.select("Model (exp_id)", EXP_OPTIONS, EXP_KEYS[0])

eval_level_s = z.select(
    "Level (CURRICULUM mode)",
    [
        ("1", "L1: Crawl"),
        ("2", "L2: Walk"),
        ("3", "L3: Trapper"),
        ("4", "L4: Locksmith"),
        ("5", "L5: Climber"),
        ("6", "L6: Scout"),
        ("7", "L7: Operative"),
        ("8", "L8: Elite"),
        ("9", "L9: Master"),
    ],
    "9",
)

ckpt_k_s = z.select("Checkpoint (k)", [(str(i), "k=%s" % i) for i in range(1, 10)], "9")

# --- Custom config fields ---
c_width = z.input("Width (custom)", "11")
c_height = z.input("Height (custom)", "11")
c_nfloors = z.input("Floors (custom)", "2")
c_hp = z.input("Agent HP (custom)", "10")
c_ammo = z.input("Agent ammo (custom)", "5")
c_max_ammo = z.input("Max ammo (custom)", "5")
c_stamina = z.input("Stamina (custom)", "3")
c_ntraps = z.input("Traps (custom)", "2")
c_nhealth = z.input("Health packs (custom)", "3")
c_nammo = z.input("Ammo packs (custom)", "1")
c_max_steps = z.input("Max steps (custom)", "200")
c_enemies = z.input("Enemies (custom, vd: patrol:0,chaser:1)", "patrol:0")
seed_str = z.input("Seed", "42")

if config_mode == "custom":
    enemies_list = []
    for part in (c_enemies or "").split(","):
        part = part.strip()
        if ":" in part:
            etype, efloor = part.split(":", 1)
            enemies_list.append(
                {"type": etype.strip(), "floor": int(efloor.strip())}
            )

    nf = int(c_nfloors or 2)
    level_cfg = {
        "name": "Custom",
        "width": int(c_width or 11),
        "height": int(c_height or 11),
        "n_floors": nf,
        "max_steps": int(c_max_steps or 200),
        "agent_hp": int(c_hp or 10),
        "agent_ammo": int(c_ammo or 5),
        "agent_max_ammo": int(c_max_ammo or 5),
        "agent_stamina": int(c_stamina or 3),
        "n_traps": int(c_ntraps or 2),
        "n_health": int(c_nhealth or 3),
        "n_ammo": int(c_nammo or 1),
        "enemies": enemies_list,
        "locks": [],
        "start_floor": nf - 1,
        "goal_floor": 0,
    }
else:
    eval_level = int(eval_level_s) if eval_level_s else 9
    level_cfg = copy.deepcopy(CURRICULUM[eval_level])

seed_str = (seed_str or "").strip()
if not seed_str:
    import random
    base_seed = random.randint(0, 2**31 - 1)
else:
    try:
        base_seed = int(seed_str)
    except ValueError:
        base_seed = hash(seed_str) % (2**31)

ckpt_k = int(ckpt_k_s or 9)

z.z.angularBind("d_gif_status", "Đang chạy...")


def run_gif_single_ckpt(exp_id, level_cfg, base_seed, ckpt_k):
    exp_id = _normalize_exp_id(exp_id)

    OUTPUT_DIR_GIF = _gif_output_dir()
    GIF_SUBDIR = os.path.join(OUTPUT_DIR_GIF, "gifs")
    os.makedirs(GIF_SUBDIR, exist_ok=True)

    ckpt_path = os.path.join(OUTPUT_DIR_GIF, "%s_level_%s.zip" % (exp_id, ckpt_k))
    if not os.path.isfile(ckpt_path):
        print("  SKIP k=%s: khong co file checkpoint" % ckpt_k)
        return

    level_name = level_cfg.get("name", "Custom")
    fname = "gif_%s_ckpt%s_seed%s.gif" % (exp_id, ckpt_k, base_seed)
    out_path = os.path.join(GIF_SUBDIR, fname)

    print(
        "Sinh 1 GIF cho: exp_id=%s | config=%s | seed=%s | ckpt_k=%s"
        % (exp_id, level_name, base_seed, ckpt_k)
    )

    t0 = time.time()
    try:
        model = load_model_for_exp(exp_id, ckpt_path)
    except Exception as e:
        print("  ERROR load ckpt_k=%s: %s" % (ckpt_k, e))
        return

    try:
        n_frames, win = record_episode_gif(model, level_cfg, base_seed, out_path)
        elapsed = time.time() - t0
        update_gif_meta(fname, n_frames, win, elapsed)
        print("  [1/1] %s | frames=%s win=%s (%.1fs)" % (fname, n_frames, win, elapsed))
    except Exception as e:
        print("  ERROR rollout ckpt_k=%s: %s" % (ckpt_k, e))


run_gif_single_ckpt(selected_exp, level_cfg, base_seed, ckpt_k)

count = load_recent_gifs_to_angular()
z.z.angularBind(
    "d_gif_status",
    "Đã nạp %d file GIF mới nhất vào danh sách chọn tự do." % count,
)

%angular
<style>
.dg-root{font-family:Arial,sans-serif;background:#ffffff;padding:16px;border-radius:8px;border:1px solid #ddd;color:#333;max-width:100%}
.dg-title{font-size:16px;font-weight:bold;margin-bottom:8px;color:#222;}
.dg-hint{font-size:12px;color:#666;margin-bottom:12px;line-height:1.5}
.dg-st{background:#f0f8ff;padding:8px 12px;border-radius:4px;border:1px solid #cce0ff;font-size:12px;margin:10px 0;text-align:center;color:#004085;font-weight:bold}
.dg-compare{display:flex;flex-wrap:wrap;gap:16px;margin-top:16px}
.dg-slot{flex:1;min-width:280px;max-width:380px;background:#fafafa;padding:12px;border-radius:6px;border:1px solid #ddd;text-align:center;}
.dg-slot-title{font-size:12px;font-weight:bold;color:#444;text-transform:uppercase;margin-bottom:8px}
.dg-select{width:100%;padding:6px;border-radius:4px;border:1px solid #ccc;background:#fff;color:#333;font-size:12px;margin-bottom:12px;outline:none;}
.dg-gif-box{min-height:80px;display:flex;flex-direction:column;align-items:center;justify-content:center}
.dg-gif-box img{border:1px solid #bbb;border-radius:4px;image-rendering:pixelated;max-width:100%;}
.dg-meta{font-size:11px;color:#555;margin-top:6px;padding:4px 8px;background:#f0f0f0;border-radius:4px;border:1px solid #eee}
.dg-no{font-size:12px;color:#888;padding:20px 0}
</style>
<div class="dg-root">
<div class="dg-title">&#9654; So sánh GIF Checkpoint đa Model</div>
<div class="dg-hint">Tất cả các file GIF trong thư mục sẽ tự động được load.</div>
<div class="dg-st">{{d_gif_status}}</div>
<div class="dg-compare">
  <div class="dg-slot" ng-repeat="slot in ['A', 'B', 'C']" ng-init="selectedGif=''">
    <div class="dg-slot-title">{{slot}}</div>
    <select class="dg-select" ng-model="selectedGif" ng-options="g.name as g.name for g in d_all_gifs">
      <option value="">-- Chọn file GIF --</option>
    </select>
    <div class="dg-gif-box">
      <div ng-repeat="g in d_all_gifs" ng-if="g.name == selectedGif">
        <img ng-src="data:image/gif;base64,{{g.b64}}" />
        <div class="dg-meta">{{g.meta}}</div>
      </div>
      <div class="dg-no" ng-if="!selectedGif">Mời chọn file</div>
    </div>
  </div>
</div>
</div>


### Phần E — Đánh giá & Phân tích lộ trình học (Evaluation & Analysis)


In [ ]:

# Mặc định KHÔNG chạy (bật RUN_EVAL cần đã chạy ô MFM_ROOT + B2 để có OUTPUT_DIR)

RUN_EVAL = False  # đổi True nếu muốn chạy lại eval

if RUN_EVAL:
    import os, json, copy
    import numpy as np
    import torch.nn as nn
    from stable_baselines3 import PPO, A2C, DQN

    EVAL_N_EPISODES = 30
    EVAL_START_SEED = 9000
    LEVELS = list(range(1, 10))
    HARD = 9
    OUT_PATH = os.path.join(OUTPUT_DIR, "eval_metrics.json")

    pol_shared_128 = dict(
        features_extractor_class=SmallDictExtractor,
        features_extractor_kwargs=dict(features_dim=128),
        net_arch=dict(pi=[128, 64], vf=[128, 64]),
        activation_fn=nn.ReLU,
        share_features_extractor=True,
    )

    def load_model_for_exp(exp_id, ckpt_path):
        dev = DEVICE if "DEVICE" in globals() else "cpu"
        if exp_id == "exp1_ppo":
            return PPO.load(ckpt_path, device=dev, custom_objects={"policy_kwargs": pol_shared_128})
        if exp_id == "exp6_dqn":
            return DQN.load(
                ckpt_path,
                device=dev,
                custom_objects={"policy_kwargs": dict(
                    features_extractor_class=SmallDictExtractor,
                    features_extractor_kwargs=dict(features_dim=128),
                    net_arch=[128, 64],
                    activation_fn=nn.ReLU,
                )},
            )
        return A2C.load(ckpt_path, device=dev, custom_objects={"policy_kwargs": pol_shared_128})

    def eval_on_level(exp_id, ckpt_level, level_cfg):
        ckpt_path = os.path.join(OUTPUT_DIR, f"{exp_id}_level_{ckpt_level}.zip")
        model = load_model_for_exp(exp_id, ckpt_path)

        rewards, regrets, wins = [], [], []
        traj_lens, dmg_taken, ammo_used = [], [], []
        episode_rewards, episode_regrets = [], []
        termination_counts = {}

        for ep in range(EVAL_N_EPISODES):
            seed = EVAL_START_SEED + ep
            env = MazeEnv(config=copy.deepcopy(level_cfg))
            obs, _ = env.reset(seed=seed)

            hp0, ammo0 = float(env.agent.hp), float(env.agent.ammo)
            done, total_r = False, 0.0
            first, lstm_state, info_end = True, None, {}

            while not done:
                if first:
                    action, lstm_state = predict_first_step(model, obs, deterministic=True)
                    first = False
                else:
                    action, lstm_state = predict_next_step(model, obs, lstm_state, deterministic=True)

                obs, r, term, trunc, info_end = env.step(action)
                total_r += float(r)
                done = bool(term or trunc)

            max_theoretical = float(getattr(env, "max_theoretical_reward", np.nan))
            regret = max_theoretical - total_r

            hp_end, ammo_end = float(env.agent.hp), float(env.agent.ammo)
            dmg = max(0.0, hp0 - hp_end)
            used = max(0.0, ammo0 - ammo_end)

            rewards.append(total_r)
            regrets.append(float(regret))
            wins.append(1.0 if info_end.get("win", False) else 0.0)
            traj_lens.append(float(info_end.get("steps", 0)))
            dmg_taken.append(float(dmg))
            ammo_used.append(float(used))

            episode_rewards.append(float(total_r))
            episode_regrets.append(float(regret))

            term_key = str(info_end.get("termination_type", "unknown"))
            termination_counts[term_key] = termination_counts.get(term_key, 0) + 1

        return {
            "win_rate": float(np.mean(wins)),
            "expected_reward": float(np.mean(rewards)),
            "regret": float(np.mean(regrets)),
            "trajectory_length": float(np.mean(traj_lens)),
            "damage_taken": float(np.mean(dmg_taken)),
            "ammo_used": float(np.mean(ammo_used)),
            "episode_rewards": episode_rewards,
            "episode_regrets": episode_regrets,
            "termination_counts": termination_counts,
        }

    exp_ids = list(EXPERIMENTS.keys())
    metrics = {
        "progression": {e: {} for e in exp_ids},
        "generalization": {e: {} for e in exp_ids},
        "forgetting": {e: {} for e in exp_ids},
        "behavioral": {e: {} for e in exp_ids},
    }

    hard_cfg = copy.deepcopy(CURRICULUM[HARD])

    for e in exp_ids:
        for k in LEVELS:
            metrics["progression"][e][str(k)] = eval_on_level(e, ckpt_level=k, level_cfg=copy.deepcopy(CURRICULUM[k]))
        for k in LEVELS:
            metrics["generalization"][e][str(k)] = eval_on_level(e, ckpt_level=k, level_cfg=hard_cfg)
        for k in LEVELS:
            metrics["forgetting"][e][str(k)] = eval_on_level(e, ckpt_level=HARD, level_cfg=copy.deepcopy(CURRICULUM[k]))
        metrics["behavioral"][e][str(HARD)] = eval_on_level(e, ckpt_level=HARD, level_cfg=hard_cfg)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print("DONE. Wrote:", OUT_PATH)

else:
    print("SKIP EVAL")

In [ ]:
# --- E-eval ---

import os, json, glob

_base = (os.environ.get("MFM_ANALYSIS_DIR") or os.environ.get("MFM_OUTPUT_DIR") or "").strip()
if not _base:
    raise RuntimeError(
        "Chưa cấu thư mục: chạy ô MFM_ROOT trước (hoặc set MFM_ANALYSIS_DIR / MFM_OUTPUT_DIR)."
    )
INPUT_DIR = os.path.abspath(os.path.expanduser(_base))
EVAL_JSON = os.path.join(INPUT_DIR, "eval_metrics.json")

print("INPUT_DIR:", INPUT_DIR)
print("EVAL_JSON:", EVAL_JSON)

if not os.path.isfile(EVAL_JSON):
    raise FileNotFoundError(
        f"Không thấy {EVAL_JSON}. Chạy ô MFM_ROOT đúng thư mục chứa eval_metrics.json (hoặc set MFM_ANALYSIS_DIR)."
    )

with open(EVAL_JSON, "r", encoding="utf-8") as f:
    EVAL_METRICS = json.load(f)

TRAIN_STATS_FILES = sorted(glob.glob(os.path.join(INPUT_DIR, "*_train_stats.json")))
TRAIN_STATS = {}
for fp in TRAIN_STATS_FILES:
    key = os.path.basename(fp).replace("_train_stats.json", "")
    try:
        with open(fp, "r", encoding="utf-8") as f:
            
            TRAIN_STATS[key] = json.load(f)
    except Exception as e:
        print("[WARN] Không đọc được", fp, "->", e)

print("Loaded eval metrics modes:", list(EVAL_METRICS.keys()))
if "progression" in EVAL_METRICS:
    print("Experiments (progression):", list(EVAL_METRICS["progression"].keys()))

print("Found train stats files:", len(TRAIN_STATS_FILES))
for k in sorted(TRAIN_STATS.keys()):
    tp = TRAIN_STATS[k].get("train_points_all", []) if isinstance(TRAIN_STATS[k], dict) else []
    print(f" - {k}: train_points_all={len(tp)}")

print("OK: Dữ liệu đã sẵn sàng cho cell E-generate-chart.")


In [ ]:
# --- E-generate-chart: vẽ biểu đồ từ JSON đã nạp (analysis-only) ---

import os, json
import numpy as np
import matplotlib.pyplot as plt

_base = (os.environ.get("MFM_ANALYSIS_DIR") or os.environ.get("MFM_OUTPUT_DIR") or "").strip()
if not _base:
    raise RuntimeError(
        "Chưa cấu thư mục: chạy ô MFM_ROOT trước (hoặc set MFM_ANALYSIS_DIR / MFM_OUTPUT_DIR)."
    )
INPUT_DIR = os.path.abspath(os.path.expanduser(_base))
EVAL_JSON = os.path.join(INPUT_DIR, "eval_metrics.json")

if "EVAL_METRICS" in globals() and isinstance(EVAL_METRICS, dict):
    M = EVAL_METRICS
else:
    assert os.path.isfile(EVAL_JSON), f"Missing {EVAL_JSON}. Run E-eval first."
    with open(EVAL_JSON, "r", encoding="utf-8") as f:
        M = json.load(f)

exp_ids = sorted(
    set(M.get("progression", {}).keys())
    | set(M.get("generalization", {}).keys())
    | set(M.get("forgetting", {}).keys())
    | set(M.get("behavioral", {}).keys())
)
if not exp_ids:
    raise ValueError("Không tìm thấy experiments trong eval_metrics.json")

mode = z.select(
    "E-mode",
    [
        ("progression", "Progression (K->K)"),
        ("generalization", "Generalization (K->9)"),
        ("forgetting", "Forgetting (9->K)"),
        ("behavioral", "Behavioral summary (@9)"),
    ],
    "progression",
)

metric = z.select(
    "Metric",
    [
        ("expected_reward", "expected_reward"),
        ("win_rate", "win_rate"),
        ("trajectory_length", "trajectory_length"),
        ("damage_taken", "damage_taken"),
        ("ammo_used", "ammo_used"),
    ],
    "expected_reward",
)

selected = z.checkbox("Experiments", [(e, e) for e in exp_ids], exp_ids)
if isinstance(selected, str):
    selected = [selected] if selected else []
if not selected:
    selected = exp_ids

LEVELS = list(range(1, 10))
HARD = 9


if mode in ("progression", "generalization", "forgetting"):
    _ = plt.figure(figsize=(10, 5))
    for exp in selected:
        bucket = M.get(mode, {}).get(exp, {})
        xs, ys = [], []
        for k in LEVELS:
            row = bucket.get(str(k))
            if isinstance(row, dict) and metric in row:
                xs.append(k)
                ys.append(float(row[metric]))
        if xs:
            _ = plt.plot(xs, ys, marker="o", linewidth=2, label=exp)

    _ = plt.title(f"{mode} - {metric}")
    _ = plt.xlabel("level")
    _ = plt.ylabel(metric)
    _ = plt.xticks(LEVELS)
    plt.grid(alpha=0.25)
    _ = plt.legend(loc="best")
    plt.tight_layout()
    plt.show()
    plt.close()

elif mode == "behavioral":
    _ = plt.figure(figsize=(10, 5))
    xs = np.arange(len(selected))
    ys = []
    for exp in selected:
        row = M.get("behavioral", {}).get(exp, {}).get(str(HARD), {})
        ys.append(float(row.get(metric, np.nan)) if isinstance(row, dict) else np.nan)

    _ = plt.bar(xs, ys)
    _ = plt.xticks(xs, selected, rotation=20, ha="right")
    _ = plt.title(f"behavioral@{HARD} - {metric}")
    _ = plt.ylabel(metric)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close()


### End